# Algorithm22 Tutorial Audit — Faithful KLDM-CPS-CSP with PyXtal and CrystalFormer

This notebook audits both the earlier Algorithm22 draft and the current updated specification. It focuses on a staged feasibility ladder before any large sampler run.

Pipeline focus:

- `Algorithm22A`: oracle-template KLDM-CPS debug projection
- `Algorithm22B-PyXtal`: exact-composition PyXtal template coverage and ranking
- `Algorithm22B-Hybrid`: PyXtal candidates with CrystalFormer scoring or lightweight q proposals
- KLDM/TDM survival after clean-space CPS projection

The design rule remains: project KLDM’s clean estimate, not the noisy KLDM state.


## CrystalFormer weights

Recommended reference links:
- [CrystalFormer available weights](https://github.com/deepmodeling/CrystalFormer#available-weights)
- [CrystalFormer CSP Colab](https://colab.research.google.com/drive/1I7b5exbB2oBjexFIEaeDQexmYRDgLHVk?authuser=0#scrollTo=kfu6Ez9e6Sp7)

The CSP-only checkpoint expected by this notebook is:
- `epoch_046000.pkl`

You can either:
- set `KLDM_ALGO21_CF_CHECKPOINT` to a local checkpoint path
- or run the download cell below


## CrystalFormer dependency bootstrap

**Purpose:** CrystalFormer needs a small JAX-side stack before the checkpoint wrapper can load. If the preflight below reports missing packages such as `haiku`, run the optional install cell once, restart the kernel, and rerun from the top.


In [1]:
# Optional dependency bootstrap for CrystalFormer.
# Run this only if the dependency preflight reports missing packages.

# %pip install -U "jax[cpu]"
# %pip install dm-haiku==0.0.15 optax==0.2.6 pyxtal==1.1.1 gdown


In [2]:
import importlib.util
import pandas as pd
rows = []
for module_name in ('jax', 'haiku', 'optax', 'pyxtal', 'gdown'):
    rows.append({
        'test': 'algorithm22_cf_dependency_preflight',
        'module': module_name,
        'available': bool(importlib.util.find_spec(module_name) is not None),
    })

dep_df = pd.DataFrame(rows)
display(dep_df)


,test,module,available
0,algorithm22_cf_dependency_preflight,jax,True
1,algorithm22_cf_dependency_preflight,haiku,True
2,algorithm22_cf_dependency_preflight,optax,True
3,algorithm22_cf_dependency_preflight,pyxtal,True
4,algorithm22_cf_dependency_preflight,gdown,True


In [3]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")
os.environ.setdefault("JAX_DISABLE_JIT", "true")
os.environ.setdefault("KLDM_ALGO21_SAFE_MODE", "true")
import time
import math
import traceback
import subprocess
import sys
import pickle
import json as jsonlib
import importlib
import gc
from dataclasses import dataclass, replace
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml
from pymatgen.core import Element

ROOT = Path.cwd().resolve()
if not ((ROOT / 'configs').exists() and (ROOT / 'src').exists()):
    if ((ROOT.parent / 'configs').exists() and (ROOT.parent / 'src').exists()):
        ROOT = ROOT.parent

import kldmPlus.symmetry.crystalformer_backend as cf_backend
import kldmPlus.algorithm22_faithful_kldm_cps_csp as algo22_backend
cf_backend = importlib.reload(cf_backend)
algo22_backend = importlib.reload(algo22_backend)

from kldmPlus.run_sampling_compare import SamplingCompareRunner, set_seed
from kldmPlus.sample_evaluation import build_structure_from_sample, evaluate_csp_reconstruction
from kldmPlus.symmetry import build_symmetry_frame_bridge, estimate_semantic_transport_for_reference_order, oracle_spacegroup_from_case
from kldmPlus.utils.time import iter_sampling_times, make_times
from kldmPlus.symmetry.wyckoff_templates import extract_wyckoff_templates, flatten_site_signature, sample_random_free_vars, recover_template_free_vars_from_anchor_entries

Algorithm19State = algo22_backend.Algorithm19State
build_oracle_diffcsppp_payload_from_structure = algo22_backend.build_oracle_diffcsppp_payload_from_structure
map_model_to_payload_reference_chart = algo22_backend.map_model_to_payload_reference_chart
map_payload_reference_chart_to_model_frame = algo22_backend.map_payload_reference_chart_to_model_frame
kldm_clean_fractional_denoiser_Df = algo22_backend.kldm_clean_fractional_denoiser_Df
wrap01 = algo22_backend.wrap01
wrapdiff = algo22_backend.wrapdiff
torus_rmse = algo22_backend.torus_rmse

CrystalFormerLikelihood = algo22_backend.CrystalFormerLikelihood
predict_clean_f0 = algo22_backend.predict_clean_f0
model_to_payload = algo22_backend.model_to_payload
payload_to_model = algo22_backend.payload_to_model
expand_q = algo22_backend.expand_q
fit_q_to_clean_prediction = algo22_backend.fit_q_to_clean_prediction
torus_soft_project = algo22_backend.torus_soft_project
renoise_from_f0 = algo22_backend.renoise_from_f0
sample_q_from_crystalformer = algo22_backend.sample_q_from_crystalformer
build_payload_from_template_q = algo22_backend.build_payload_from_template_q
species_match_reorder = algo22_backend.species_match_reorder
Algorithm21Config = cf_backend.Algorithm21Config
q_only_clean_cf_fit = cf_backend.q_only_clean_cf_fit
q_only_clean_cf_local_rerank = cf_backend.q_only_clean_cf_local_rerank
torus_interp = cf_backend.torus_interp
rank_q_candidates = getattr(cf_backend, 'rank_q_candidates', None)
build_crystalformer_reduced_sequence = getattr(cf_backend, 'build_crystalformer_reduced_sequence', None)
expand_crystalformer_reduced_sequence = getattr(cf_backend, 'expand_crystalformer_reduced_sequence', None)
crystalformer_reduced_sequence_debug = getattr(cf_backend, 'crystalformer_reduced_sequence_debug', None)
crystalformer_site_representative_search = getattr(cf_backend, 'crystalformer_site_representative_search', None)
crystalformer_payload_order_assembly_debug = getattr(cf_backend, 'crystalformer_payload_order_assembly_debug', None)

CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

PROFILE = os.environ.get('KLDM_ALGO21_PROFILE', 'laptop').strip().lower()
def profile_default(name: str, laptop: str, deep: str | None = None) -> str:
    if name in os.environ:
        return os.environ[name]
    if PROFILE in {'full', 'deep'} and deep is not None:
        return deep
    return laptop

def notebook_log(message: str) -> None:
    text = str(message)
    print(text)
    with ALGO21_LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(text)
        if not text.endswith("\n"):
            handle.write("\n")

def cf_nll_eval(*, payload, q, lattice_feature, formula, label: str) -> float:
    notebook_log(f"[cf-eval] start {label}")
    if not CF_READY:
        notebook_log(f"[cf-eval] skip {label} CF unavailable")
        return float('nan')
    use_subprocess = str(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS', 'true')).strip().lower() in {'1', 'true', 'yes', 'on'}
    try:
        q_np = np.asarray(torch.as_tensor(q).detach().cpu(), dtype=float)
        l_np = np.asarray(torch.as_tensor(lattice_feature).detach().cpu(), dtype=float)
        if use_subprocess:
            tmp_dir = ROOT / 'notebooks' / '.algo21_cf_eval_tmp'
            tmp_dir.mkdir(parents=True, exist_ok=True)
            safe_label = ''.join(ch if ch.isalnum() or ch in {'-', '_'} else '_' for ch in str(label))[-120:]
            in_path = tmp_dir / f'{safe_label}.pkl'
            out_path = tmp_dir / f'{safe_label}.json'
            with in_path.open('wb') as handle:
                pickle.dump({
                    'repo_root': str(ROOT),
                    'checkpoint_path': str(CF_CHECKPOINT),
                    'symmetry_payload': payload,
                    'q': q_np,
                    'lattice_feature': l_np,
                    'formula': formula,
                }, handle)
            cmd = [sys.executable, str(ROOT / 'scripts' / 'algorithm22_cf_score_once.py'), '--input', str(in_path), '--output', str(out_path)]
            notebook_log(f"[cf-eval] subprocess start {label}")
            completed = subprocess.run(cmd, cwd=str(ROOT), text=True, capture_output=True, timeout=int(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS_TIMEOUT', '900')), start_new_session=True)
            result = {}
            if out_path.exists():
                with out_path.open('r', encoding='utf-8') as handle:
                    result = jsonlib.load(handle)
            if completed.returncode != 0 or not result.get('ok'):
                notebook_log(f"[cf-eval] subprocess ERROR {label} returncode={completed.returncode} stderr_tail={completed.stderr[-500:]}")
                raise RuntimeError(result.get('error_message') or completed.stderr[-500:] or 'CrystalFormer subprocess failed')
            value = float(result['value'])
        else:
            raise RuntimeError('In-kernel CrystalFormer evaluation is disabled for Algorithm22 notebook stability.')
        notebook_log(f"[cf-eval] done {label} value={value:.6g}")
        return value
    except Exception as exc:
        notebook_log(f"[cf-eval] ERROR {label} {type(exc).__name__}: {exc}")
        raise
    finally:
        notebook_cleanup()

def cf_sequence_smoke(*, payload, q, lattice_feature, label: str) -> dict[str, Any]:
    notebook_log(f"[cf-seq] start {label}")
    seq = build_crystalformer_reduced_sequence(payload=payload, q=q, lattice_feature=lattice_feature)
    notebook_log(f"[cf-seq] done {label} num_sites={int(seq['num_sites'])}")
    notebook_cleanup()
    return seq

def parse_int_list(text: str) -> list[int]:
    return [int(v.strip()) for v in str(text).split(',') if v.strip()]

def parse_float_list(text: str) -> list[float]:
    return [float(v.strip()) for v in str(text).split(',') if v.strip()]

SAMPLE_SEED = int(profile_default('KLDM_ALGO21_SEED', '2', '2'))
GRAPH_IDS = parse_int_list(profile_default('KLDM_ALGO21_GRAPH_IDS', '2', '2'))
CORE_T_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_CORE_T', '0.1,0.3,0.5', '0.1,0.3,0.5')))
BETA_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_BETA_VALUES', '0,0.001,0.01,0.1,1,10', '0,0.001,0.01,0.1,1,10')))
ALPHA_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_ALPHA_VALUES', '0,0.1,0.25,0.5,1.0', '0,0.1,0.25,0.5,1.0')))
Q_OPT_STEPS = int(profile_default('KLDM_ALGO21_Q_OPT_STEPS', '50', '50'))
TEST45_BETA_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_TEST45_BETA_VALUES', '0', '0')))
TEST6_ALPHA_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_TEST6_ALPHA_VALUES', '0,0.25', '0,0.25')))
TEST46_Q_OPT_STEPS = int(profile_default('KLDM_ALGO21_TEST46_Q_OPT_STEPS', '4', '4'))
TEST46_CORE_T_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_TEST46_CORE_T', '0.5', '0.5')))
TEST7_RERANK_RADIUS = float(profile_default('KLDM_ALGO21_TEST7_RERANK_RADIUS', '0.05', '0.05'))
TEST7_RERANK_CANDIDATES = int(profile_default('KLDM_ALGO21_TEST7_RERANK_CANDIDATES', '3', '3'))
TEST7_RERANK_KEEP_TOL = float(profile_default('KLDM_ALGO21_TEST7_RERANK_KEEP_TOL', '0.05', '0.05'))
TEST45_CF_SCALE_BETAS = tuple(parse_float_list(profile_default('KLDM_ALGO21_TEST45_CF_SCALE_BETAS', '0,1e-6,1e-5,1e-4', '0,1e-6,1e-5,1e-4')))
# CF-gradient optimization is deliberately opt-in; default notebook tests use value-only reranking.
os.environ.setdefault('KLDM_ALGO21_ENABLE_CF_GRAD', 'false')
SHORT_SAMPLER_STEPS = int(profile_default('KLDM_ALGO21_SHORT_SAMPLER_STEPS', '100', '100'))
SHORT_SAMPLER_TSTART = float(profile_default('KLDM_ALGO21_SHORT_SAMPLER_TSTART', '0.5', '0.5'))
NOTEBOOK_SAFE_MODE = str(os.environ.get("KLDM_ALGO21_SAFE_MODE", "true")).strip().lower() in {"1", "true", "yes", "on"}
SHOW_FULL_DEBUG_TABLES = not NOTEBOOK_SAFE_MODE
CF_RANDOM_TRIALS = int(profile_default('KLDM_ALGO21_CF_RANDOM_TRIALS', '1' if NOTEBOOK_SAFE_MODE else '8', '1'))
CF_CHECKPOINT = str(profile_default('KLDM_ALGO21_CF_CHECKPOINT', str((ROOT / 'notebooks' / 'epoch_046000.pkl') if (ROOT / 'notebooks' / 'epoch_046000.pkl').exists() else (ROOT / 'epoch_046000.pkl')), str((ROOT / 'notebooks' / 'epoch_046000.pkl') if (ROOT / 'notebooks' / 'epoch_046000.pkl').exists() else (ROOT / 'epoch_046000.pkl'))))
CF_FORMULA = os.environ.get('KLDM_ALGO21_FORMULA', '').strip() or None
ALGO21_LOG_PATH = ROOT / 'notebooks' / 'log.txt'
os.environ["KLDM_ALGO21_LOG_PATH"] = str(ALGO21_LOG_PATH)
ALGO21_LOG_PATH.write_text("", encoding="utf-8")

runner = SamplingCompareRunner(config_path=CONFIG_PATH)
runner.model.eval()
set_seed(SAMPLE_SEED)

requested_num_targets = max(max(GRAPH_IDS) if GRAPH_IDS else 0, len(GRAPH_IDS), 5)
if int(runner.compare_cfg.get('num_targets', 0)) < requested_num_targets:
    runner.compare_cfg['num_targets'] = int(requested_num_targets)
    runner.compare_cfg['batch_size'] = max(int(runner.compare_cfg.get('batch_size', 0)), int(requested_num_targets))
    runner.loader, runner.lattice_transform = runner._build_loader()
full_batch = next(iter(runner.loader)).to(runner.device)
full_ptr = full_batch.ptr.tolist()
full_num_graphs = max(len(full_ptr) - 1, 0)
available_graph_ids = [int(graph_id) for graph_id in GRAPH_IDS if 1 <= int(graph_id) <= full_num_graphs]
selected_graph_indices0 = [int(graph_id) - 1 for graph_id in available_graph_ids]
selected_items = full_batch.index_select(selected_graph_indices0) if hasattr(full_batch, 'index_select') else full_batch[selected_graph_indices0]
if isinstance(selected_items, list):
    batch = full_batch.__class__.from_data_list(selected_items)
else:
    batch = selected_items
batch = batch.to(runner.device)
ptr = batch.ptr.detach().cpu().tolist()

print(f'profile={PROFILE} graphs={GRAPH_IDS} t={CORE_T_VALUES} beta={BETA_VALUES} alpha={ALPHA_VALUES} q_opt_steps={Q_OPT_STEPS} cf_ckpt={CF_CHECKPOINT}')


def notebook_cleanup():
    use_subprocess = str(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS', 'true')).strip().lower() in {'1', 'true', 'yes', 'on'}
    if use_subprocess:
        try:
            cf_obj = globals().get('cf_like', None)
            if cf_obj is not None and hasattr(cf_obj, 'release_runtime'):
                cf_obj.release_runtime()
        except Exception:
            pass
    gc.collect()
    try:
        import jax
        jax.clear_caches()
    except Exception:
        pass
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

Algorithm22Config = algo22_backend.Algorithm22Config
algorithm22_projection_schedule = algo22_backend.algorithm22_projection_schedule
algorithm22_piecewise_alpha = algo22_backend.algorithm22_piecewise_alpha
algorithm22_cps_gamma = algo22_backend.algorithm22_cps_gamma
algorithm22_cps_alpha = algo22_backend.algorithm22_cps_alpha
clean_fractional_estimate_22 = algo22_backend.clean_fractional_estimate
clean_lattice_estimate_22 = algo22_backend.clean_lattice_estimate
decode_state_cell_matrix_22 = algo22_backend.decode_state_cell_matrix
oracle_template_projection_22 = algo22_backend.oracle_template_projection
generate_candidates_22 = algo22_backend.generate_candidates
generate_pyxtal_candidates_22 = algo22_backend.generate_pyxtal_candidates
rank_candidates_against_clean_estimate_22 = algo22_backend.rank_candidates_against_clean_estimate
algorithm22b_ranked_kldm_cps_step = algo22_backend.algorithm22b_ranked_kldm_cps_step
group_witness_from_candidates_22 = algo22_backend.group_witness_from_candidates


Algorithm22RunResult = algo22_backend.Algorithm22RunResult
algorithm22A_oracle_template_kldm_cps = algo22_backend.algorithm22A_oracle_template_kldm_cps
algorithm22B_ranked_kldm_cps = algo22_backend.algorithm22B_ranked_kldm_cps
kldm_clean_fractional_denoiser_22 = algo22_backend.kldm_clean_fractional_denoiser
kldm_clean_lattice_denoiser_22 = algo22_backend.kldm_clean_lattice_denoiser
expand_template_to_model_order_22 = algo22_backend.expand_template_to_model_order
materialize_candidate_from_template_22 = algo22_backend.materialize_candidate_from_template
rank_projected_candidates_with_rule_22 = algo22_backend.rank_projected_candidates_with_rule
sample_template_q_proposals_22 = algo22_backend.sample_template_q_proposals
safety_ok_22 = algo22_backend.safety_ok
torus_distance_squared_22 = algo22_backend.torus_distance_squared
oracle_template_witness_22 = algo22_backend.oracle_template_witness
project_candidates_to_clean_estimate_22 = algo22_backend.project_candidates_to_clean_estimate
algorithm22_piecewise_alpha = algo22_backend.algorithm22_piecewise_alpha
algorithm22_piecewise_beta = algo22_backend.algorithm22_piecewise_beta
kldm_cps_update_or_renoise_22 = algo22_backend.kldm_cps_update_or_renoise
tdm_velocity_sigma_at_state_22 = algo22_backend.tdm_velocity_sigma_at_state
tdm_residual_epsilon_r_22 = algo22_backend.tdm_residual_epsilon_r


MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen
mps:  False
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
profile=laptop graphs=[2] t=(0.1, 0.3, 0.5) beta=(0.0, 0.001, 0.01, 0.1, 1.0, 10.0) alpha=(0.0, 0.1, 0.25, 0.5, 1.0) q_opt_steps=50 cf_ckpt=/workspace/notebooks/epoch_

In [4]:
@dataclass
class GraphCase:
    graph_id: int
    graph_idx0: int
    composition: dict[int, int]
    atomic_numbers: torch.Tensor
    gt_frac: torch.Tensor
    gt_l_feature: torch.Tensor
    requested_sg: int

result_tables: dict[str, pd.DataFrame] = {}
payload_cache: dict[int, Any] = {}


def dataframe_result(name: str, rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if 'PASS' not in df.columns:
        df['PASS'] = False
    if 'status' not in df.columns:
        df['status'] = np.where(df['PASS'], 'PASS', 'FAIL')
    result_tables[name] = df
    return df


def safe_display_sorted(df: pd.DataFrame, by: list[str]):
    df = df.copy()
    cols = list(df.columns)
    for key in by:
        if key not in cols:
            df[key] = np.nan
    display(df.sort_values(by).reset_index(drop=True))


def error_row(test: str, graph: int | str, exc: Exception, failure_category: str, **extra) -> dict[str, Any]:
    tb = traceback.format_exc().strip().splitlines()
    return {
        'test': test,
        'graph': graph,
        'PASS': False,
        'status': 'ERROR',
        'error_type': type(exc).__name__,
        'error_message': str(exc),
        'traceback_tail': tb[-1] if tb else '',
        'failure_category': failure_category,
        **extra,
    }


def graph_slice(graph_idx0: int) -> tuple[int, int]:
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def composition_counter(values) -> dict[int, int]:
    arr = [int(v) for v in torch.as_tensor(values, dtype=torch.long).reshape(-1).tolist()]
    from collections import Counter
    return dict(sorted(Counter(arr).items()))


def graph_edge_node_index(case: GraphCase) -> torch.Tensor:
    start, end = graph_slice(case.graph_idx0)
    edge_index = batch.edge_node_index
    mask = ((edge_index[0] >= start) & (edge_index[0] < end) & (edge_index[1] >= start) & (edge_index[1] < end))
    return (edge_index[:, mask] - start).detach().clone()


def graph_tensors(graph_idx0: int) -> dict[str, Any]:
    start, end = graph_slice(graph_idx0)
    return {
        'pos': batch.pos[start:end].detach().clone(),
        'l': batch.l[graph_idx0].detach().clone().reshape(-1),
        'h': batch.atomic_numbers[start:end].detach().clone().to(torch.long),
        'sg': int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
    }


def load_test_graphs(graph_ids=available_graph_ids) -> list[GraphCase]:
    out = []
    for graph_idx0, graph_id in enumerate(graph_ids):
        g = graph_tensors(graph_idx0)
        out.append(GraphCase(
            graph_id=int(graph_id),
            graph_idx0=int(graph_idx0),
            composition=composition_counter(g['h']),
            atomic_numbers=g['h'],
            gt_frac=g['pos'],
            gt_l_feature=g['l'],
            requested_sg=g['sg'],
        ))
    return out

GRAPH_CASES = load_test_graphs(available_graph_ids)
print('loaded_graphs=', [g.graph_id for g in GRAPH_CASES], 'sg=', [g.requested_sg for g in GRAPH_CASES])


def make_single_graph_batch_view(case: GraphCase, *, ref_tensor: torch.Tensor | None = None):
    device = case.gt_frac.device if ref_tensor is None else ref_tensor.device
    dtype = case.gt_frac.dtype if ref_tensor is None else ref_tensor.dtype
    pos = case.gt_frac.detach().clone().to(device=device, dtype=dtype)
    batch_index = torch.zeros(pos.shape[0], device=device, dtype=torch.long)
    num_atoms = torch.tensor([int(pos.shape[0])], device=device, dtype=torch.long)
    edge_node_index = graph_edge_node_index(case).to(device=device)
    return SimpleNamespace(
        pos=pos,
        batch=batch_index,
        num_atoms=num_atoms,
        num_graphs=1,
        edge_node_index=edge_node_index,
    )


def build_case_structure(case: GraphCase):
    return build_structure_from_sample(case.gt_frac, case.gt_l_feature, case.atomic_numbers, lattice_transform=runner.lattice_transform)


def build_oracle_payload(case: GraphCase):
    if case.graph_id in payload_cache:
        return payload_cache[case.graph_id]
    structure = build_case_structure(case)
    payload = build_oracle_diffcsppp_payload_from_structure(
        standardized_structure=structure,
        requested_spacegroup=oracle_spacegroup_from_case(case),
        tol=1e-2,
    )
    bridge = build_symmetry_frame_bridge(vanilla_structure=structure, standardization='refined', symprec=1e-2, angle_tolerance=5.0)
    tau_ref, assignment_ref, rmse_ref = estimate_semantic_transport_for_reference_order(
        standardized_reference_frac_coords=np.asarray(payload.expanded_frac_coords, dtype=float),
        standardized_reference_atomic_numbers=np.asarray(payload.expanded_atomic_numbers, dtype=int),
        bridge=bridge,
    )
    linear_std_to_model = np.asarray(bridge.standardized_to_vanilla_linear, dtype=float)
    linear_model_to_std = np.linalg.inv(linear_std_to_model)
    payload = replace(
        payload,
        debug_info={
            **(payload.debug_info or {}),
            'model_reference_frac_coords': np.asarray(structure.frac_coords, dtype=float).tolist(),
            'model_to_payload_linear': linear_model_to_std.tolist(),
            'model_to_payload_tau': np.asarray(tau_ref, dtype=float).tolist(),
            'model_to_payload_order': np.asarray(assignment_ref, dtype=int).tolist(),
            'payload_to_model_linear': linear_std_to_model.tolist(),
            'payload_to_model_tau': np.asarray(tau_ref, dtype=float).tolist(),
            'payload_to_model_order': np.asarray(assignment_ref, dtype=int).tolist(),
            'transport_rmse': float(rmse_ref),
        },
    )
    payload_cache[case.graph_id] = payload
    return payload


def evaluate_arrays(case: GraphCase, pred_f: torch.Tensor, pred_l: torch.Tensor, pred_h: torch.Tensor) -> dict[str, Any]:
    result = evaluate_csp_reconstruction(
        pred_f=pred_f,
        pred_l=pred_l,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_l_feature,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
        requested_space_group=case.requested_sg,
        validity_cutoff=0.5,
    )
    standardized_frac_rmse = None
    if getattr(result, 'matcher_diagnostics', None) is not None:
        standardized_frac_rmse = getattr(result.matcher_diagnostics, 'standardized_frac_rmse', None)
    return {
        'match': bool(result.match),
        'valid': bool(result.valid),
        'rmse': result.rmse,
        'frac_rmse': result.frac_rmse,
        'composition_match': result.composition_match,
        'requested_space_group_match': result.requested_space_group_match,
        'validity_reason': result.validity_reason,
        'standardized_frac_rmse': standardized_frac_rmse,
        'metric_source': 'sample_evaluation.evaluate_csp_reconstruction / StructureMatcher',
    }


def algo_times(case: GraphCase, t_value: float):
    dtype = case.gt_frac.dtype
    device = case.gt_frac.device
    t_graph = torch.as_tensor([[float(t_value)]], device=device, dtype=dtype)
    t_nodes = torch.full((int(case.gt_frac.shape[0]),), float(t_value), device=device, dtype=dtype)
    return t_graph, t_nodes


def make_algo_state_from_raw(*, f, v, l, atom_types, node_index, edge_node_index, t_graph, t_nodes) -> Algorithm19State:
    return Algorithm19State(
        f=f.detach().clone(),
        v=v.detach().clone(),
        l=l.detach().clone().reshape(-1),
        atom_types=atom_types.detach().clone(),
        node_index=node_index.detach().clone(),
        edge_node_index=edge_node_index.detach().clone(),
        t_graph=t_graph.detach().clone(),
        t_nodes=t_nodes.detach().clone(),
    )


def clean_prediction(state: Algorithm19State, *, variant: str = 'minus', coordinate_score_mode: str = 'direct') -> torch.Tensor:
    return kldm_clean_fractional_denoiser_Df(
        model=runner.model,
        f=state.f,
        v=state.v,
        l=state.l,
        atom_types=state.atom_types,
        t_graph=state.t_graph,
        t_nodes=state.t_nodes,
        node_index=state.node_index,
        edge_index=state.edge_node_index,
        variant=variant,
        coordinate_score_mode=coordinate_score_mode,
    )


def sample_gt_noisy_state(case: GraphCase, *, t_value: float, seed: int = SAMPLE_SEED):
    batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
    set_seed(int(seed) + 1000 * int(case.graph_id) + int(round(1000.0 * float(t_value))))
    t_graph, t_nodes = algo_times(case, float(t_value))
    f_t, v_t, *_ = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=case.gt_frac, index=batch_view.batch)
    state = make_algo_state_from_raw(
        f=f_t,
        v=v_t,
        l=case.gt_l_feature,
        atom_types=case.atomic_numbers,
        node_index=batch_view.batch,
        edge_node_index=batch_view.edge_node_index,
        t_graph=t_graph,
        t_nodes=t_nodes,
    )
    return state


def sample_template_q(case: GraphCase, payload, *, mode: str, seed: int = SAMPLE_SEED, dtype=None, device=None):
    chart = algo22_backend._get_wyckoff_dof_chart(payload)
    dtype = case.gt_frac.dtype if dtype is None else dtype
    device = case.gt_frac.device if device is None else device
    if chart.total_dof == 0:
        return torch.empty((0,), device=device, dtype=dtype)
    mode_norm = str(mode).strip().lower()
    if mode_norm in {'crystalformer_oracle', 'oracle_structure', 'oracle'}:
        return torch.as_tensor(chart.q_ref, device=device, dtype=dtype).reshape(-1)
    if mode_norm in {'random_template', 'random'}:
        set_seed(int(seed) + 30000 * int(case.graph_id))
        return torch.rand((int(chart.total_dof),), device=device, dtype=dtype)
    raise ValueError(f'Unsupported q mode {mode!r}')


def build_template_f0_model(case: GraphCase, payload, *, q0: torch.Tensor):
    chart = algo22_backend._get_wyckoff_dof_chart(payload)
    q_eval = torch.remainder(q0.reshape(-1), 1.0) if q0.numel() else q0.reshape(-1)
    z_payload = chart.expand_q(q_eval, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    f0_model = map_payload_reference_chart_to_model_frame(z_payload, payload)
    return f0_model.detach().clone(), z_payload.detach().clone()


def sample_template_noisy_state(case: GraphCase, *, init_mode: str, t_value: float, seed: int = SAMPLE_SEED):
    payload = build_oracle_payload(case)
    q0 = sample_template_q(case, payload, mode=init_mode, seed=seed, dtype=case.gt_frac.dtype, device=case.gt_frac.device)
    f0_model, z0_payload = build_template_f0_model(case, payload, q0=q0)
    batch_view = make_single_graph_batch_view(case, ref_tensor=f0_model)
    set_seed(int(seed) + 70000 * int(case.graph_id) + int(round(1000.0 * float(t_value))) + (0 if init_mode == 'crystalformer_oracle' else 1))
    t_graph, t_nodes = algo_times(case, float(t_value))
    f_t, v_t, *_ = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=f0_model, index=batch_view.batch)
    state = make_algo_state_from_raw(
        f=f_t,
        v=v_t,
        l=case.gt_l_feature,
        atom_types=case.atomic_numbers,
        node_index=batch_view.batch,
        edge_node_index=batch_view.edge_node_index,
        t_graph=t_graph,
        t_nodes=t_nodes,
    )
    return state, payload, q0, z0_payload


def q_distance(a: torch.Tensor, b: torch.Tensor) -> float:
    if a.numel() == 0 and b.numel() == 0:
        return 0.0
    diff = wrapdiff(a.reshape(-1), b.reshape(-1))
    return float(torch.sqrt(diff.square().mean()).detach().item())


def composition_to_formula_string(composition: dict[int, int]) -> str:
    counts = [int(v) for v in composition.values() if int(v) > 0]
    gcd_value = 0
    for value in counts:
        gcd_value = int(value) if gcd_value == 0 else math.gcd(int(gcd_value), int(value))
    gcd_value = gcd_value or 1
    parts = []
    for atomic_number in sorted(int(k) for k in composition):
        count = int(composition[atomic_number])
        if count <= 0:
            continue
        reduced = count // gcd_value
        symbol = Element.from_Z(int(atomic_number)).symbol
        parts.append(symbol if reduced == 1 else f'{symbol}{reduced}')
    return ''.join(parts)


def cf_formula_for_case(case: GraphCase) -> str | None:
    return CF_FORMULA or composition_to_formula_string(case.composition)


def config21(*, beta: float, alpha: float, q_opt_steps: int = Q_OPT_STEPS, post_renoise_acceptance: bool = True, debug_prints: bool = False, cf_grad_max_dims: int | None = None, cf_value_only_after_renoise: bool = False) -> Algorithm21Config:
    return Algorithm21Config(
        beta=float(beta),
        alpha=float(alpha),
        q_opt_steps=int(q_opt_steps),
        post_renoise_acceptance=bool(post_renoise_acceptance),
        debug_prints=bool(debug_prints),
        cf_grad_max_dims=cf_grad_max_dims,
        cf_value_only_after_renoise=bool(cf_value_only_after_renoise),
    )


def fit_clean_q(clean_f: torch.Tensor, *, case: GraphCase, payload, config: Algorithm21Config, cf_like=None, formula: str | None = None, q_init: torch.Tensor | None = None):
    z_payload = map_model_to_payload_reference_chart(clean_f, payload)
    return q_only_clean_cf_fit(
        z_payload=z_payload,
        payload=payload,
        t_nodes=torch.full((int(clean_f.shape[0]),), 0.3, device=clean_f.device, dtype=clean_f.dtype),
        lattice_feature=case.gt_l_feature,
        formula=formula,
        config=config,
        cf_likelihood=cf_like,
        q_init=q_init,
    )


loaded_graphs= [2] sg= [4]


## Audit Map

This notebook cross-checks the Algorithm22 implementation against the local code we rely on and records coverage for both the earlier markdown draft and the current update.


In [5]:
audit_rows = [
    {'spec_version': 'old_algorithm22_markdown', 'area': 'A-F', 'status': 'audited', 'note': 'Used as baseline for earlier backend/notebook draft.'},
    {'spec_version': 'current_algorithm22_update', 'area': 'A-F', 'status': 'audited', 'note': 'This notebook tracks the current requested tests and implementation status.'},
]
source_rows = [
    {'source': 'src/PPR/ppr-main/ppr/projection.py', 'verified_point': 'CPS/PPR project-clean-then-continue principle.'},
    {'source': 'src/PPR/ppr-main/ppr/diffusion/samplers.py', 'verified_point': 'Projection lives inside the reverse-step flow after prediction.'},
    {'source': 'src/kldmPlus/symmetry/pcs_projection.py', 'verified_point': 'PyXtal-backed exact-composition template states already exist.'},
    {'source': 'src/CrystalFormer/CrystalFormer-main/crystalformer/src/sample.py', 'verified_point': 'CrystalFormer reduced-variable sampling remains expensive and is treated carefully.'},
    {'source': 'src/CrystalFormer/CrystalFormer-main/crystalformer/src/loss.py', 'verified_point': 'CrystalFormer coordinate NLL is used as a score, not the hard symmetry constraint.'},
]
symbol_rows = []
for name in [
    'algorithm22_projection_schedule', 'algorithm22_piecewise_alpha', 'algorithm22A_oracle_template_kldm_cps',
    'algorithm22B_ranked_kldm_cps', 'generate_candidates_22', 'sample_template_q_proposals_22',
    'expand_template_to_model_order_22', 'rank_projected_candidates_with_rule_22', 'safety_ok_22',
]:
    symbol_rows.append({'backend_symbol': name, 'available': bool(name in globals())})
display(pd.DataFrame(audit_rows))
display(pd.DataFrame(source_rows))
display(pd.DataFrame(symbol_rows))


,spec_version,area,status,note
0,old_algorithm22_markdown,A-F,audited,Used as baseline for earlier backend/notebook ...
1,current_algorithm22_update,A-F,audited,This notebook tracks the current requested tes...


,source,verified_point
0,src/PPR/ppr-main/ppr/projection.py,CPS/PPR project-clean-then-continue principle.
1,src/PPR/ppr-main/ppr/diffusion/samplers.py,Projection lives inside the reverse-step flow ...
2,src/kldmPlus/symmetry/pcs_projection.py,PyXtal-backed exact-composition template state...
3,src/CrystalFormer/CrystalFormer-main/crystalfo...,CrystalFormer reduced-variable sampling remain...
4,src/CrystalFormer/CrystalFormer-main/crystalfo...,CrystalFormer coordinate NLL is used as a scor...


,backend_symbol,available
0,algorithm22_projection_schedule,True
1,algorithm22_piecewise_alpha,True
2,algorithm22A_oracle_template_kldm_cps,True
3,algorithm22B_ranked_kldm_cps,True
4,generate_candidates_22,True
5,sample_template_q_proposals_22,True
6,expand_template_to_model_order_22,True
7,rank_projected_candidates_with_rule_22,True
8,safety_ok_22,True


## Runtime Preflight

Load the KLDM comparison runner and, when available, the CrystalFormer checkpoint wrapper used for hybrid scoring and approximate CrystalFormer-only diagnostics.


In [6]:
rows = []
CF_READY = False
CF_COORDINATE_ONLY = np.nan
notebook_log('[algo22.preflight] cf checkpoint load start')
try:
    ckpt_exists = Path(CF_CHECKPOINT).exists()
    if ckpt_exists:
        tmp_dir = ROOT / 'notebooks' / '.algo21_cf_eval_tmp'
        tmp_dir.mkdir(parents=True, exist_ok=True)
        out_path = tmp_dir / 'algorithm22_cf_checkpoint_smoke.json'
        cmd = [
            sys.executable,
            str(ROOT / 'scripts' / 'algorithm22_cf_checkpoint_smoke.py'),
            '--repo-root', str(ROOT),
            '--checkpoint', str(CF_CHECKPOINT),
            '--output', str(out_path),
        ]
        completed = subprocess.run(cmd, cwd=str(ROOT), text=True, capture_output=True, timeout=int(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS_TIMEOUT', '900')), start_new_session=True)
        result = {}
        if out_path.exists():
            with out_path.open('r', encoding='utf-8') as handle:
                result = jsonlib.load(handle)
        CF_READY = bool(completed.returncode == 0 and result.get('ok'))
        CF_COORDINATE_ONLY = bool(result.get('coordinate_only')) if CF_READY else np.nan
        if completed.returncode != 0 and not CF_READY:
            notebook_log(f"[algo22.preflight] cf checkpoint smoke failed stderr_tail={completed.stderr[-500:]}")
    rows.append({
        'test': 'algorithm22_cf_checkpoint_load',
        'checkpoint_path': CF_CHECKPOINT,
        'checkpoint_exists': bool(ckpt_exists),
        'wrapper_ready': bool(CF_READY),
        'coordinate_only': CF_COORDINATE_ONLY,
        'PASS': bool(CF_READY),
        'status': 'INFO' if CF_READY else ('MISSING_CHECKPOINT' if not ckpt_exists else 'SUBPROCESS_SMOKE_FAILED'),
    })
except Exception as exc:
    rows.append(error_row('algorithm22_cf_checkpoint_load', 'na', exc, 'cf_checkpoint_load', checkpoint_path=CF_CHECKPOINT))
safe_display_sorted(dataframe_result('algorithm22_cf_checkpoint_load', rows), ['checkpoint_path'])
notebook_cleanup()


[algo22.preflight] cf checkpoint load start


,test,checkpoint_path,checkpoint_exists,wrapper_ready,coordinate_only,PASS,status
0,algorithm22_cf_checkpoint_load,/workspace/notebooks/epoch_046000.pkl,True,True,True,True,INFO


In [7]:

ALGO22_CFG = Algorithm22Config(
    schedule=algo22_backend.Algorithm22ScheduleConfig(
        n_pc_steps=800,
        projection_interval=50,
        p_start=0.625,
        alpha_max=0.25,
    ),
    top_branches=1,
    state_return_mode='preserve_velocity_shift',
    lattice_projection=False,
    post_acceptance=True,
)
ALGO22_SCHEDULE = algorithm22_projection_schedule(schedule=ALGO22_CFG.schedule)
ALGO22_PROJECTION_STEPS = [int(pt.step) for pt in ALGO22_SCHEDULE if pt.project]
ALGO22_LOCAL_STEPS = [600, 750]
RUN_ALGO22_CF_HEAVY = str(os.environ.get('RUN_ALGO22_CF_HEAVY', 'true')).strip().lower() in {'1', 'true', 'yes', 'on'}
ALGO22_PHASEA_SOURCES = ['baseline', 'oracle_template', 'pyxtal_only'] + (['pyxtal_cf_score'] if RUN_ALGO22_CF_HEAVY and CF_READY else [])
ALGO22_PHASEE_SOURCES = ['baseline', 'oracle_template', 'pyxtal_only'] + (['pyxtal_cf_score', 'pyxtal_cf_q_augmented'] if RUN_ALGO22_CF_HEAVY and CF_READY else [])


def algo22_step_to_progress(step: int, n_pc_steps: int = 800) -> float:
    return float(step) / float(max(1, n_pc_steps - 1))


def algo22_step_to_t(step: int, n_pc_steps: int = 800) -> float:
    return float(max(1.0e-6, 1.0 - algo22_step_to_progress(step, n_pc_steps=n_pc_steps)))


def oracle_signature(case: GraphCase):
    payload = build_oracle_payload(case)
    return tuple(sorted((int(z), str(label)) for z, label in zip(payload.anchor_atomic_numbers.tolist(), payload.wyckoff_letters)))


def template_signature(template) -> tuple:
    return flatten_site_signature(template)


def candidate_rmse_to_gt(case: GraphCase, frac_coords_model_order: torch.Tensor) -> float:
    return float(evaluate_arrays(case, frac_coords_model_order, case.gt_l_feature, case.atomic_numbers)['frac_rmse'])


def safe_metric_float(value, default: float = float('nan')) -> float:
    if value is None:
        return float(default)
    try:
        out = float(value)
    except Exception:
        return float(default)
    return out if np.isfinite(out) else float(default) if not np.isnan(default) else out


def safe_metric_bool(value) -> bool:
    return bool(False if value is None else value)


def eval_metric_bundle(ev: dict[str, Any]) -> dict[str, Any]:
    return {
        'frac_rmse': safe_metric_float(ev.get('frac_rmse')),
        'rmse': safe_metric_float(ev.get('rmse')),
        'match': safe_metric_bool(ev.get('match')),
        'valid': safe_metric_bool(ev.get('valid')),
        'composition_match': safe_metric_bool(ev.get('composition_match')),
        'requested_space_group_match': safe_metric_bool(ev.get('requested_space_group_match')),
        'standardized_frac_rmse': safe_metric_float(ev.get('standardized_frac_rmse')),
    }


def topk_min(values, k: int):
    vals = [float(v) for v in values if np.isfinite(float(v))]
    if not vals:
        return float('nan')
    return float(min(sorted(vals)[: max(1, int(k))]))


ALGO22_CF_SCORE_CANDIDATE_LIMIT = int(os.environ.get('ALGO22_CF_SCORE_CANDIDATE_LIMIT', '4'))
ALGO22_CF_AUG_BASE_LIMIT = int(os.environ.get('ALGO22_CF_AUG_BASE_LIMIT', '1'))


def cf_nll_eval_batch(*, items, label: str) -> list[float]:
    if not CF_READY or not items:
        return [float('nan')] * len(items)
    tmp_dir = ROOT / 'notebooks' / '.algo21_cf_eval_tmp'
    tmp_dir.mkdir(parents=True, exist_ok=True)
    safe_label = ''.join(ch if ch.isalnum() or ch in {'-', '_'} else '_' for ch in str(label))[-120:]
    in_path = tmp_dir / f'{safe_label}_batch.pkl'
    out_path = tmp_dir / f'{safe_label}_batch.json'
    packed = []
    for item in items:
        packed.append({
            'symmetry_payload': item['payload'],
            'q': np.asarray(torch.as_tensor(item['q']).detach().cpu(), dtype=float),
            'lattice_feature': np.asarray(torch.as_tensor(item['lattice_feature']).detach().cpu(), dtype=float),
            'formula': item.get('formula'),
        })
    with in_path.open('wb') as handle:
        pickle.dump({
            'repo_root': str(ROOT),
            'checkpoint_path': str(CF_CHECKPOINT),
            'items': packed,
        }, handle)
    cmd = [sys.executable, str(ROOT / 'scripts' / 'algorithm22_cf_score_batch_once.py'), '--input', str(in_path), '--output', str(out_path)]
    notebook_log(f'[cf-eval-batch] subprocess start {label} n={len(items)}')
    completed = subprocess.run(cmd, cwd=str(ROOT), text=True, capture_output=True, timeout=int(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS_TIMEOUT', '900')), start_new_session=True)
    result = {}
    if out_path.exists():
        with out_path.open('r', encoding='utf-8') as handle:
            result = jsonlib.load(handle)
    if completed.returncode != 0 or not result.get('ok'):
        notebook_log(f"[cf-eval-batch] ERROR {label} returncode={completed.returncode} stderr_tail={completed.stderr[-500:]}")
        raise RuntimeError(result.get('error_message') or completed.stderr[-500:] or 'CrystalFormer batch subprocess failed')
    values = [float(v) for v in result.get('values', [])]
    if len(values) != len(items):
        raise RuntimeError(f'CrystalFormer batch returned {len(values)} values for {len(items)} items')
    return values


def cf_formula_for_case(case: GraphCase) -> str | None:
    return CF_FORMULA or composition_to_formula_string(case.composition)


def q_strategy_label(strategy: str) -> str:
    mapping = {
        'pcs_anchor': 'pcs_anchor',
        'uniform': 'uniform',
        'sobol': 'sobol',
        'special-position-biased': 'special',
        'local': 'local',
        'small_local_perturbations': 'local',
    }
    return mapping.get(str(strategy), str(strategy))


def candidate_cf_stats(candidates, *, cf_mode: str):
    cf_vals = [float(c.cf_nll) for c in candidates if np.isfinite(float(c.cf_nll))]
    sampled = sum(1 for c in candidates if 'cf_sample_index' in (c.metadata or {}))
    return {
        'cf_active': bool(CF_READY and cf_mode != 'none'),
        'cf_mode': str(cf_mode),
        'cf_nll_finite': bool(len(cf_vals) > 0),
        'num_cf_q_samples': int(sampled),
        'num_cf_scored': int(len(cf_vals)),
        'cf_nll_min': float(np.min(cf_vals)) if cf_vals else float('nan'),
        'cf_nll_max': float(np.max(cf_vals)) if cf_vals else float('nan'),
        'cf_nll_has_nan': bool(any(not np.isfinite(float(c.cf_nll)) for c in candidates)),
    }


def make_algo22_cfg(*, p_start: float | None = None, q_sampling_strategies=None, num_q_per_template: int | None = None, top_branches: int | None = None, cf_sample_k: int | None = None, source_mode: str | None = None, state_return_mode: str | None = None):
    cfg = ALGO22_CFG
    if p_start is not None:
        cfg = replace(cfg, schedule=replace(cfg.schedule, p_start=float(p_start)))
    if q_sampling_strategies is not None:
        cfg = replace(cfg, pyxtal_q_sampling_strategies=tuple(q_sampling_strategies))
    if num_q_per_template is not None:
        cfg = replace(cfg, pyxtal_q_samples_per_template=int(num_q_per_template))
    if top_branches is not None:
        cfg = replace(cfg, top_branches=int(top_branches))
    if cf_sample_k is not None:
        cfg = replace(cfg, cf_sample_k=int(cf_sample_k))
    if source_mode is not None:
        cfg = replace(cfg, crystalformer_template_mode=str(source_mode))
    if state_return_mode is not None:
        cfg = replace(cfg, state_return_mode=str(state_return_mode))
    return cfg


def sample_q_candidates_eval(*, payload, lattice_feature, formula, K: int, seed: int, top_p: float = 1.0, temperature: float = 1.0):
    if not CF_READY:
        return [], []
    use_subprocess = str(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS', 'true')).strip().lower() in {'1', 'true', 'yes', 'on'}
    if use_subprocess:
        tmp_dir = ROOT / 'notebooks' / '.algo21_cf_eval_tmp'
        tmp_dir.mkdir(parents=True, exist_ok=True)
        in_path = tmp_dir / f'algorithm22_sample_q_K{int(K)}_seed{int(seed)}.pkl'
        out_path = tmp_dir / f'algorithm22_sample_q_K{int(K)}_seed{int(seed)}.json'
        with in_path.open('wb') as handle:
            pickle.dump({
                'repo_root': str(ROOT),
                'checkpoint_path': str(CF_CHECKPOINT),
                'symmetry_payload': payload,
                'lattice_feature': np.asarray(torch.as_tensor(lattice_feature).detach().cpu(), dtype=float),
                'formula': formula,
                'K': int(K),
                'top_p': float(top_p),
                'temperature': float(temperature),
                'seed': int(seed),
            }, handle)
        cmd = [sys.executable, str(ROOT / 'scripts' / 'algorithm22_cf_sample_q_once.py'), '--input', str(in_path), '--output', str(out_path)]
        completed = subprocess.run(cmd, cwd=str(ROOT), text=True, capture_output=True, timeout=int(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS_TIMEOUT', '900')), start_new_session=True)
        result = {}
        if out_path.exists():
            with out_path.open('r', encoding='utf-8') as handle:
                result = jsonlib.load(handle)
        if completed.returncode != 0 or not result.get('ok'):
            raise RuntimeError(result.get('error_message') or completed.stderr[-500:] or 'CrystalFormer q sampling subprocess failed')
        q_candidates = [torch.as_tensor(q_i, dtype=torch.as_tensor(lattice_feature).dtype, device=torch.as_tensor(lattice_feature).device) for q_i in result['q_candidates']]
        cf_nll = [float(v) for v in result.get('cf_nll', [])]
        return q_candidates, cf_nll
    return sample_q_from_crystalformer(
        payload=payload,
        lattice_feature=torch.as_tensor(lattice_feature),
        formula=formula,
        cf_likelihood=None,
        K=int(K),
        top_p=float(top_p),
        temperature=float(temperature),
        seed=int(seed),
    )


def generate_source_candidates(case: GraphCase, state, f0_hat: torch.Tensor, *, source: str, cfg):
    formula = cf_formula_for_case(case)
    cf_mode = 'none'
    if source in {'pyxtal_cf_score', 'pyxtal_cf_q_augmented'} and not RUN_ALGO22_CF_HEAVY:
        return tuple(), {
            'cf_active': bool(CF_READY),
            'cf_mode': 'disabled',
            'cf_nll_finite': False,
            'num_cf_q_samples': 0,
            'num_cf_scored': 0,
            'cf_nll_min': float('nan'),
            'cf_nll_max': float('nan'),
            'cf_nll_has_nan': False,
        }
    if source == 'pyxtal_only':
        candidates = generate_candidates_22(
            f0_hat=f0_hat,
            state=state,
            space_group=case.requested_sg,
            source='pyxtal_only',
            lattice_transform=runner.lattice_transform,
            config=cfg,
            formula=formula,
            cf_likelihood=None,
            debug_label=f'g={case.graph_id} src={source}',
        )
        candidates = tuple(candidates[: max(1, int(ALGO22_PYXTAL_CANDIDATE_LIMIT))])
        return candidates, candidate_cf_stats(candidates, cf_mode=cf_mode)
    if source == 'pyxtal_cf_score':
        cf_mode = 'score_only'
        base_candidates = generate_candidates_22(
            f0_hat=f0_hat,
            state=state,
            space_group=case.requested_sg,
            source='pyxtal_only',
            lattice_transform=runner.lattice_transform,
            config=cfg,
            formula=formula,
            cf_likelihood=None,
            debug_label=f'g={case.graph_id} src=pyxtal_only_for_cf_score',
        )
        base_candidates = tuple(base_candidates[: max(1, int(ALGO22_CF_SCORE_CANDIDATE_LIMIT))])
        nll_values = cf_nll_eval_batch(items=[
            {'payload': cand.payload, 'q': cand.q_init, 'lattice_feature': cand.lattice_feature, 'formula': formula}
            for cand in base_candidates
        ], label=f'g{case.graph_id}_cfscore_batch') if base_candidates else []
        scored = []
        for cand, nll in zip(base_candidates, nll_values):
            scored.append(replace(cand, source='pyxtal_cf_score', cf_nll=float(nll), metadata={**(cand.metadata or {}), 'cf_scored': True}))
        candidates = tuple(scored)
        return candidates, candidate_cf_stats(candidates, cf_mode=cf_mode)
    if source == 'pyxtal_cf_q_augmented':
        cf_mode = 'q_augmented'
        base_candidates = generate_candidates_22(
            f0_hat=f0_hat,
            state=state,
            space_group=case.requested_sg,
            source='pyxtal_only',
            lattice_transform=runner.lattice_transform,
            config=cfg,
            formula=formula,
            cf_likelihood=None,
            debug_label=f'g={case.graph_id} src=pyxtal_only_for_cf_aug',
        )
        augmented = []
        for base_idx, cand in enumerate(base_candidates[: min(len(base_candidates), int(ALGO22_CF_AUG_BASE_LIMIT))]):
            augmented.append(replace(cand, source='pyxtal_cf_q_augmented', metadata={**(cand.metadata or {}), 'cf_aug_base': True}))
            q_candidates, cf_nll_values = sample_q_candidates_eval(
                payload=cand.payload,
                lattice_feature=cand.lattice_feature,
                formula=formula,
                K=int(cfg.cf_sample_k),
                top_p=float(cfg.cf_top_p),
                temperature=float(cfg.cf_temperature),
                seed=int(cfg.cf_sampler_seed) + int(case.graph_id) * 1000 + int(base_idx),
            )
            for sample_idx, (q_i, nll) in enumerate(zip(q_candidates, cf_nll_values)):
                try:
                    aug = materialize_candidate_from_template_22(
                        template=cand.template,
                        q=q_i.reshape(-1),
                        state=state,
                        lattice_matrix=cand.lattice_matrix,
                        lattice_feature=cand.lattice_feature,
                        source='pyxtal_cf_q_augmented',
                        metadata={**(cand.metadata or {}), 'cf_sample_index': int(sample_idx), 'cf_aug_base_index': int(base_idx)},
                        cf_nll=float(nll),
                        spacegroup=case.requested_sg,
                    )
                    augmented.append(aug)
                except Exception:
                    continue
        candidates = tuple(augmented)
        return candidates, candidate_cf_stats(candidates, cf_mode=cf_mode)
    raise ValueError(f'Unsupported source: {source}')


def fit_candidate_to_clean(case: GraphCase, state, f0_hat: torch.Tensor, candidate, cfg):
    fitted = algo22_backend.fit_q_to_template(
        candidate=candidate,
        target_f0=f0_hat,
        target_atomic_numbers=state.atom_types,
        q_init=candidate.q_init,
        lambda_init=0.0,
        q_opt_steps=int(cfg.q_opt_steps),
        q_lr=float(cfg.q_lr),
        grad_clip=float(cfg.grad_clip),
    )
    return fitted


ALGO22_LOCAL_SOURCES = ['no_correction', 'oracle_template', 'pyxtal_only'] + (['pyxtal_cf_score', 'pyxtal_cf_q_augmented'] if RUN_ALGO22_CF_HEAVY and CF_READY else [])
ALGO22_PYXTAL_CANDIDATE_LIMIT = int(os.environ.get('ALGO22_PYXTAL_CANDIDATE_LIMIT', '8'))
REPLAY_CACHE = {}


def fit_oracle_payload_from_q_init(case: GraphCase, state, f0_hat: torch.Tensor, payload, *, q_init, cfg, init_label: str):
    q_gt = torch.as_tensor(algo22_backend._get_wyckoff_dof_chart(payload).q_ref, device=f0_hat.device, dtype=f0_hat.dtype).reshape(-1)
    z_hat = model_to_payload(f_model=f0_hat, payload=payload)
    q_init_tensor = None if q_init is None else torch.as_tensor(q_init, device=f0_hat.device, dtype=f0_hat.dtype).reshape(-1)
    fit = fit_q_to_clean_prediction(
        z_hat=z_hat,
        payload=payload,
        t_nodes=state.t_nodes,
        lattice_feature=state.l,
        q_init=q_init_tensor,
        q_init_mode='random' if q_init_tensor is None else 'oracle_structure',
        steps=int(cfg.q_opt_steps),
        lr=float(cfg.q_lr),
        grad_clip=float(cfg.grad_clip),
    )
    f_proj = payload_to_model(z_payload=fit.z_proj_payload, payload=payload)
    ev = evaluate_arrays(case, f_proj, case.gt_l_feature, case.atomic_numbers)
    q_init_dist = float('nan') if q_init_tensor is None or q_init_tensor.numel() != q_gt.numel() else float(torch.sqrt(cf_backend.torus_mse(q_init_tensor.reshape(-1), q_gt.reshape(-1)).clamp_min(0.0)).detach().item())
    q_fit_dist = float('nan') if fit.q_star.numel() != q_gt.numel() else float(torch.sqrt(cf_backend.torus_mse(fit.q_star.reshape(-1), q_gt.reshape(-1)).clamp_min(0.0)).detach().item())
    return {
        'init_label': str(init_label),
        'q_init_distance_to_GT': q_init_dist,
        'q_fit_distance_to_GT': q_fit_dist,
        'witness_after': float(fit.witness_sin),
        'frac_rmse': safe_metric_float(ev.get('frac_rmse')),
        'rmse': safe_metric_float(ev.get('rmse')),
        'match': safe_metric_bool(ev.get('match')),
        'valid': safe_metric_bool(ev.get('valid')),
    }


def ranked_summary(case: GraphCase, ranked) -> dict[str, Any]:
    if not ranked:
        return {
            'top1_frac_rmse': float('nan'),
            'top1_rmse': float('nan'),
            'top1_match': False,
            'top1_valid': False,
            'top3_min_frac_rmse': float('nan'),
        }
    top1_eval = evaluate_arrays(case, ranked[0].frac_coords_model_order, case.gt_l_feature, case.atomic_numbers)
    top3_min = min([candidate_rmse_to_gt(case, item.frac_coords_model_order) for item in ranked[:3]], default=float('nan'))
    return {
        'top1_frac_rmse': float(top1_eval['frac_rmse']),
        'top1_rmse': float(top1_eval['rmse']),
        'top1_match': bool(top1_eval['match']),
        'top1_valid': bool(top1_eval['valid']),
        'top3_min_frac_rmse': float(top3_min),
    }


def replay_source(case: GraphCase, *, source: str, cfg, alpha_override=None):
    cache_key = (int(case.graph_id), str(source), repr(cfg), None if alpha_override is None else float(alpha_override))
    if cache_key in REPLAY_CACHE:
        return REPLAY_CACHE[cache_key]
    rows = []
    baseline_rows = []
    payload = build_oracle_payload(case)
    for step in ALGO22_PROJECTION_STEPS:
        state = sample_gt_noisy_state(case, t_value=algo22_step_to_t(step))
        f0_hat = clean_fractional_estimate_22(state=state, model=runner.model, config=cfg)
        base_eval = evaluate_arrays(case, f0_hat, case.gt_l_feature, case.atomic_numbers)
        base_witness = float(oracle_template_witness_22(state=state, model=runner.model, payload=payload, config=cfg))
        baseline_rows.append({'step': step, 'frac_rmse': safe_metric_float(base_eval.get('frac_rmse')), 'rmse': safe_metric_float(base_eval.get('rmse')), 'match': safe_metric_bool(base_eval.get('match')), 'valid': safe_metric_bool(base_eval.get('valid')), 'witness': base_witness})
        alpha = float(algorithm22_piecewise_alpha(algo22_step_to_progress(step)) if alpha_override is None else alpha_override)
        if source == 'baseline':
            rows.append({**baseline_rows[-1], 'accepted': True, 'cf_active': False, 'cf_mode': 'none', 'cf_nll_finite': False, 'num_cf_q_samples': 0})
            continue
        if source == 'oracle_template':
            branch = oracle_template_projection_22(state=state, model=runner.model, payload=payload, alpha=alpha, beta=1.0, config=cfg)
            ev = evaluate_arrays(case, branch.f0_hat_after, case.gt_l_feature, case.atomic_numbers)
            rows.append({'step': step, 'frac_rmse': safe_metric_float(ev.get('frac_rmse')), 'rmse': safe_metric_float(ev.get('rmse')), 'match': safe_metric_bool(ev.get('match')), 'valid': safe_metric_bool(ev.get('valid')), 'witness': float(branch.witness_after), 'accepted': bool(branch.accepted), 'cf_active': False, 'cf_mode': 'none', 'cf_nll_finite': False, 'num_cf_q_samples': 0})
            continue
        candidates, cf_stats = generate_source_candidates(case, state, f0_hat, source=source, cfg=cfg)
        ranked = rank_candidates_against_clean_estimate_22(f0_hat=f0_hat, state=state, candidates=candidates, config=cfg)
        if not ranked:
            rows.append({'step': step, 'frac_rmse': float('nan'), 'rmse': float('nan'), 'match': False, 'valid': False, 'witness': float('inf'), 'accepted': False, **cf_stats})
            continue
        branch = algo22_backend.branch_survival_step(state=state, model=runner.model, projected=ranked[0], alpha=alpha, beta=1.0, candidate_pool=tuple(candidates), config=cfg)
        ev = evaluate_arrays(case, branch.f0_hat_after, case.gt_l_feature, case.atomic_numbers)
        rows.append({'step': step, 'frac_rmse': safe_metric_float(ev.get('frac_rmse')), 'rmse': safe_metric_float(ev.get('rmse')), 'match': safe_metric_bool(ev.get('match')), 'valid': safe_metric_bool(ev.get('valid')), 'witness': float(branch.witness_after), 'accepted': bool(branch.accepted), **cf_stats})
    baseline_frac = float(np.nanmean([r['frac_rmse'] for r in baseline_rows])) if baseline_rows else float('nan')
    baseline_rmse = float(np.nanmean([r['rmse'] for r in baseline_rows])) if baseline_rows else float('nan')
    baseline_witness = float(np.nanmean([r['witness'] for r in baseline_rows])) if baseline_rows else float('nan')
    finite_rows = [r for r in rows if np.isfinite(float(r['frac_rmse']))]
    best_row = min(finite_rows, key=lambda r: float(r['frac_rmse'])) if finite_rows else None
    out = {
        'rows': rows,
        'baseline_rows': baseline_rows,
        'frac_rmse': float(np.nanmean([r['frac_rmse'] for r in rows])) if rows else float('nan'),
        'rmse': float(np.nanmean([r['rmse'] for r in rows])) if rows else float('nan'),
        'match_rate': float(np.mean([bool(r['match']) for r in rows])) if rows else float('nan'),
        'valid_rate': float(np.mean([bool(r['valid']) for r in rows])) if rows else float('nan'),
        'group_witness': float(np.nanmean([r['witness'] for r in rows])) if rows else float('nan'),
        'accepted_fraction': float(np.mean([bool(r['accepted']) for r in rows])) if rows else float('nan'),
        'projection_count': int(len(rows)),
        'best_step': int(best_row['step']) if best_row is not None else -1,
        'best_step_frac_rmse': float(best_row['frac_rmse']) if best_row is not None else float('nan'),
        'best_step_rmse_to_GT': float(best_row['rmse']) if best_row is not None else float('nan'),
        'best_step_match': bool(best_row['match']) if best_row is not None else False,
        'best_step_valid': bool(best_row['valid']) if best_row is not None else False,
        'beats_baseline': bool(np.nanmean([r['frac_rmse'] for r in rows]) < baseline_frac) if rows and np.isfinite(baseline_frac) else False,
        'improves_witness': bool(np.nanmean([r['witness'] for r in rows]) < baseline_witness) if rows and np.isfinite(baseline_witness) else False,
        'improves_frac_rmse': bool(np.nanmean([r['frac_rmse'] for r in rows]) < baseline_frac) if rows and np.isfinite(baseline_frac) else False,
        'improves_rmse': bool(np.nanmean([r['rmse'] for r in rows]) < baseline_rmse) if rows and np.isfinite(baseline_rmse) else False,
    }
    REPLAY_CACHE[cache_key] = out
    return out


Metric naming follows `sample_evaluation.evaluate_csp_reconstruction(...)` and `StructureMatcher`: `rmse` is the matcher RMS distance in Cartesian space, `frac_rmse` is the fractional-coordinate RMSE, `standardized_frac_rmse` comes from matcher diagnostics when available, and `q_*_distance_to_GT` refers to reduced Wyckoff-coordinate distance rather than structure-level error.


In [8]:

rows_a = []
for case in GRAPH_CASES:
    baseline = replay_source(case, source='baseline', cfg=ALGO22_CFG)
    for source in ALGO22_PHASEA_SOURCES:
        replay = baseline if source == 'baseline' else replay_source(case, source=source, cfg=ALGO22_CFG)
        rows_a.append({
            'test': 'algorithm22_phase_a_default_replay',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'source': source,
            'p_start': float(ALGO22_CFG.schedule.p_start),
            'projection_steps': ','.join(str(v) for v in ALGO22_PROJECTION_STEPS),
            'mean_frac_rmse': float(replay['frac_rmse']),
            'mean_rmse_to_GT': float(replay['rmse']),
            'match_rate': float(replay['match_rate']),
            'valid_rate': float(replay['valid_rate']),
            'group_witness': float(replay['group_witness']),
            'accepted_fraction': float(replay['accepted_fraction']),
            'projection_count': int(replay['projection_count']),
            'improves_witness': bool(replay['improves_witness']),
            'improves_frac_rmse': bool(replay['improves_frac_rmse']),
            'improves_rmse': bool(replay['improves_rmse']),
            'beats_baseline': bool(replay['beats_baseline']) if source != 'baseline' else False,
            'PASS': True,
            'status': 'INFO',
        })

df_a = dataframe_result('algorithm22_phase_a_default_replay', rows_a)
safe_display_sorted(df_a, ['graph', 'source'])


/workspace/src/kldmPlus/sample_evaluation/sample_evaluation.py:482: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:206.)
  return torch.as_tensor(


[cf-eval-batch] subprocess start g2_cfscore_batch n=1
[cf-eval-batch] subprocess start g2_cfscore_batch n=1
[cf-eval-batch] subprocess start g2_cfscore_batch n=1
[cf-eval-batch] subprocess start g2_cfscore_batch n=1
[cf-eval-batch] subprocess start g2_cfscore_batch n=1
[cf-eval-batch] subprocess start g2_cfscore_batch n=1


,test,graph,space_group,source,p_start,projection_steps,mean_frac_rmse_to_GT,mean_cart_rmse_to_GT,match_rate,valid_rate,group_witness,accepted_fraction,projection_count,improves_witness,improves_frac_rmse,improves_cart_rmse,beats_baseline,PASS,status
0,algorithm22_phase_a_default_replay,2,4,baseline,0.625,"500,550,600,650,700,750",0.004296,0.018215,1.0,1.0,0.003006,1.000000,6,False,False,False,False,True,INFO
1,algorithm22_phase_a_default_replay,2,4,oracle_template,0.625,"500,550,600,650,700,750",0.004437,0.018731,1.0,1.0,0.000036,1.000000,6,True,False,False,False,True,INFO
2,algorithm22_phase_a_default_replay,2,4,pyxtal_cf_score,0.625,"500,550,600,650,700,750",0.004291,0.018072,1.0,1.0,0.001221,0.666667,6,True,True,True,True,True,INFO
3,algorithm22_phase_a_default_replay,2,4,pyxtal_only,0.625,"500,550,600,650,700,750",0.004291,0.018072,1.0,1.0,0.001221,0.666667,6,True,True,True,True,True,INFO


## Phase B
PyXtal feasibility: vary q proposal strategy and `num_q_per_template`, and measure whether raw q diversity matters before and after q-fit to the KLDM clean estimate.


In [9]:
rows_b = []
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    q_gt = torch.as_tensor(algo22_backend._get_wyckoff_dof_chart(payload).q_ref, device=case.gt_frac.device, dtype=case.gt_frac.dtype).reshape(-1)
    state = sample_gt_noisy_state(case, t_value=algo22_step_to_t(600))
    f0_hat = clean_fractional_estimate_22(state=state, model=runner.model, config=ALGO22_CFG)
    for strategy in ['pcs_anchor', 'sobol', 'local']:
        for num_q_per_template in [1, 8, 32]:
            cfg = make_algo22_cfg(q_sampling_strategies=(strategy,), num_q_per_template=num_q_per_template)
            candidates, _ = generate_source_candidates(case, state, f0_hat, source='pyxtal_only', cfg=cfg)
            raw_q_rmse = [float(torch.sqrt(cf_backend.torus_mse(c.q_init.reshape(-1), q_gt.reshape(-1)).clamp_min(0.0)).detach().item()) for c in candidates if c.q_init.numel() == q_gt.numel()]
            fitted_records = []
            for cand in candidates[: min(len(candidates), 6)]:
                fitted = fit_candidate_to_clean(case, state, f0_hat, cand, cfg)
                ev = evaluate_arrays(case, fitted.frac_coords_model_order, case.gt_l_feature, case.atomic_numbers)
                fitted_records.append({'frac_rmse': safe_metric_float(ev.get('frac_rmse')), 'rmse': safe_metric_float(ev.get('rmse')), 'match': safe_metric_bool(ev.get('match')), 'valid': safe_metric_bool(ev.get('valid')), 'requested_space_group_match': safe_metric_bool(ev.get('requested_space_group_match')), 'standardized_frac_rmse': safe_metric_float(ev.get('standardized_frac_rmse'))})
            best_fitted = min(fitted_records, key=lambda x: x['frac_rmse']) if fitted_records else None
            rows_b.append({
                'test': 'algorithm22_phase_b_pyxtal_feasibility',
                'graph': case.graph_id,
                'space_group': case.requested_sg,
                'q_strategy': q_strategy_label(strategy),
                'num_q_per_template': int(num_q_per_template),
                'num_templates': int(len({template_signature(c.template) for c in candidates if c.template is not None})),
                'oracle_template_found': bool(any(template_signature(c.template) == oracle_signature(case) for c in candidates if c.template is not None)),
                'min_raw_q_rmse_to_GT': float(topk_min(raw_q_rmse, 1)),
                'top3_raw_q_rmse_to_GT': float(topk_min(raw_q_rmse, 3)),
                'top10_raw_q_rmse_to_GT': float(topk_min(raw_q_rmse, 10)),
                'after_qfit_top1_rmse_to_GT': float(min([r['frac_rmse'] for r in fitted_records], default=float('nan'))),
                'after_qfit_top3_rmse_to_GT': float(topk_min([r['frac_rmse'] for r in fitted_records], 3)),
                'best_after_qfit_rmse_to_GT': float(best_fitted['rmse']) if best_fitted is not None else float('nan'),
                'best_after_qfit_match': bool(best_fitted['match']) if best_fitted is not None else False,
                'best_after_qfit_valid': bool(best_fitted['valid']) if best_fitted is not None else False,
                'best_after_qfit_requested_space_group_match': bool(best_fitted.get('requested_space_group_match', False)) if best_fitted is not None else False,
                'runtime': float('nan'),
                'PASS': bool(len(candidates) > 0),
                'status': 'INFO',
            })

df_b = dataframe_result('algorithm22_phase_b_pyxtal_feasibility', rows_b)
safe_display_sorted(df_b, ['graph', 'q_strategy', 'num_q_per_template'])


,test,graph,space_group,q_strategy,num_q_per_template,num_templates,oracle_template_found,min_raw_q_rmse_to_GT,top3_raw_q_rmse_to_GT,top10_raw_q_rmse_to_GT,after_qfit_top1_rmse_to_GT,after_qfit_top3_rmse_to_GT,best_after_qfit_cart_rmse_to_GT,best_after_qfit_match,best_after_qfit_valid,runtime,PASS,status
0,algorithm22_phase_b_pyxtal_feasibility,2,4,local,1,1,True,0.278563,0.278563,0.278563,0.012318,0.012318,0.053832,True,True,NaN,True,INFO
1,algorithm22_phase_b_pyxtal_feasibility,2,4,local,8,1,True,0.253722,0.253722,0.253722,0.011107,0.011107,0.046613,True,True,NaN,True,INFO
2,algorithm22_phase_b_pyxtal_feasibility,2,4,local,32,1,True,0.244045,0.244045,0.244045,0.010212,0.010212,0.043240,True,True,NaN,True,INFO
3,algorithm22_phase_b_pyxtal_feasibility,2,4,pcs_anchor,1,1,True,0.313783,0.313783,0.313783,0.012154,0.012154,0.055459,True,True,NaN,True,INFO
4,algorithm22_phase_b_pyxtal_feasibility,2,4,pcs_anchor,8,1,True,0.303025,0.303025,0.303025,0.013534,0.013534,0.061040,True,True,NaN,True,INFO
5,algorithm22_phase_b_pyxtal_feasibility,2,4,pcs_anchor,32,1,True,0.275942,0.275942,0.275942,0.080625,0.080625,NaN,False,False,NaN,True,INFO
6,algorithm22_phase_b_pyxtal_feasibility,2,4,sobol,1,1,True,0.276674,0.276674,0.276674,0.011922,0.011922,0.051747,True,True,NaN,True,INFO
7,algorithm22_phase_b_pyxtal_feasibility,2,4,sobol,8,1,True,0.264652,0.264652,0.264652,0.011163,0.011163,0.049838,True,True,NaN,True,INFO
8,algorithm22_phase_b_pyxtal_feasibility,2,4,sobol,32,1,True,0.264652,0.264652,0.264652,0.011163,0.011163,0.049838,True,True,NaN,True,INFO


## Phase C
CrystalFormer feasibility: first test CF score on a richer PyXtal candidate set, then test CF q augmentation, and always log whether CF was truly active.


In [10]:
rows_c1 = []
rows_c2 = []
rows_c3 = []
rows_c4 = []
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    q_gt = torch.as_tensor(algo22_backend._get_wyckoff_dof_chart(payload).q_ref, device=case.gt_frac.device, dtype=case.gt_frac.dtype).reshape(-1)
    state = sample_gt_noisy_state(case, t_value=algo22_step_to_t(600))
    f0_hat = clean_fractional_estimate_22(state=state, model=runner.model, config=ALGO22_CFG)
    pyxtal_ref_cfg = make_algo22_cfg(q_sampling_strategies=('pcs_anchor', 'sobol', 'local'), num_q_per_template=16, top_branches=5)
    pyxtal_ref_candidates, _ = generate_source_candidates(case, state, f0_hat, source='pyxtal_only', cfg=pyxtal_ref_cfg)
    pyxtal_ref_ranked = rank_candidates_against_clean_estimate_22(f0_hat=f0_hat, state=state, candidates=pyxtal_ref_candidates, config=pyxtal_ref_cfg)
    cf_score_candidates = tuple()
    cf_score_ranked = tuple()
    cf_aug_candidates = tuple()
    if RUN_ALGO22_CF_HEAVY and CF_READY:
        for num_q in [16]:
            cfg_score = make_algo22_cfg(q_sampling_strategies=('pcs_anchor', 'uniform', 'sobol', 'special-position-biased'), num_q_per_template=num_q, top_branches=5, source_mode='score_only')
            candidates, cf_stats = generate_source_candidates(case, state, f0_hat, source='pyxtal_cf_score', cfg=cfg_score)
            cf_score_candidates = candidates
            scored = project_candidates_to_clean_estimate_22(candidates=tuple(candidates), target_f0=f0_hat, target_atomic_numbers=state.atom_types, target_l0=clean_lattice_estimate_22(state=state), config=cfg_score)
            rmse_by_id = {id(item): candidate_rmse_to_gt(case, item.frac_coords_model_order) for item in scored}
            eval_by_id = {id(item): evaluate_arrays(case, item.frac_coords_model_order, case.gt_l_feature, case.atomic_numbers) for item in scored}
            gt_vals = [rmse_by_id[id(item)] for item in scored]
            geom_vals = [float(item.witness) for item in scored]
            cf_vals = [float(item.cf_nll) for item in scored if np.isfinite(float(item.cf_nll))]
            corr_cf = float(pd.Series(cf_vals).rank().corr(pd.Series(gt_vals[:len(cf_vals)]).rank(), method='spearman')) if len(cf_vals) > 1 else float('nan')
            corr_geom = float(pd.Series(geom_vals).rank().corr(pd.Series(gt_vals).rank(), method='spearman')) if len(geom_vals) > 1 else float('nan')
            best_gt = float(min(gt_vals)) if gt_vals else float('nan')
            rank_cf = -1
            rank_geom = -1
            if np.isfinite(best_gt):
                cf_sorted = sorted(scored, key=lambda x: (float('inf') if not np.isfinite(float(x.cf_nll)) else float(x.cf_nll), rmse_by_id[id(x)]))
                geom_sorted = sorted(scored, key=lambda x: float(x.witness))
                for idx, item in enumerate(cf_sorted, start=1):
                    if abs(rmse_by_id[id(item)] - best_gt) < 1e-8:
                        rank_cf = idx
                        break
                for idx, item in enumerate(geom_sorted, start=1):
                    if abs(rmse_by_id[id(item)] - best_gt) < 1e-8:
                        rank_geom = idx
                        break
            for rule in ['geometry_only', 'geometry_first_cf_tiebreak', 'cf_only']:
                mapped_rule = {'geometry_only':'geometry_only','cf_only':'cf_only','geometry_first_cf_tiebreak':'geometry_first_cf_tiebreak','geometry_plus_small_cf':'geometry_first_cf_tiebreak'}[rule]
                ranked = rank_projected_candidates_with_rule_22(scored=scored, rule=mapped_rule, top_branches=5, eps_rank=cfg_score.eps_rank)
                cf_score_ranked = ranked if rule == 'geometry_first_cf_tiebreak' else cf_score_ranked
                top_eval = eval_by_id[id(ranked[0])] if ranked else {'frac_rmse': float('nan'), 'rmse': float('nan'), 'match': False, 'valid': False}
                rows_c1.append({
                    'test': 'algorithm22_phase_c1_cf_score_only',
                    'graph': case.graph_id,
                    'space_group': case.requested_sg,
                    'num_q_per_template': int(num_q),
                    'ranking_rule': rule,
                    'top1_frac_rmse': safe_metric_float(top_eval.get('frac_rmse')),
                    'top1_rmse_to_GT': safe_metric_float(top_eval.get('rmse')),
                    'top1_match': safe_metric_bool(top_eval.get('match')),
                    'top1_valid': safe_metric_bool(top_eval.get('valid')),
                    'top3_min_frac_rmse': float(topk_min([rmse_by_id[id(item)] for item in ranked], 3)),
                    'rank_of_best_GT_candidate_by_CF': int(rank_cf),
                    'rank_of_best_GT_candidate_by_geometry': int(rank_geom),
                    'spearman_cf_nll_vs_GT_RMSE': float(corr_cf),
                    'spearman_geometry_witness_vs_GT_RMSE': float(corr_geom),
                    **cf_stats,
                    'PASS': bool(len(candidates) > 0),
                    'status': 'INFO',
                })
        for cf_k in [4]:
            cfg_aug = make_algo22_cfg(q_sampling_strategies=('pcs_anchor',), num_q_per_template=1, cf_sample_k=cf_k, source_mode='q_augmented')
            candidates, cf_stats = generate_source_candidates(case, state, f0_hat, source='pyxtal_cf_q_augmented', cfg=cfg_aug)
            cf_aug_candidates = candidates
            raw_eval = [evaluate_arrays(case, c.frac_coords_model_order, case.gt_l_feature, case.atomic_numbers) for c in candidates]
            fitted_eval = []
            for cand in candidates[: min(len(candidates), 6)]:
                fitted = fit_candidate_to_clean(case, state, f0_hat, cand, cfg_aug)
                ev = evaluate_arrays(case, fitted.frac_coords_model_order, case.gt_l_feature, case.atomic_numbers)
                fitted_eval.append(ev)
            best_after = min(fitted_eval, key=lambda x: x['frac_rmse']) if fitted_eval else None
            rows_c2.append({
                'test': 'algorithm22_phase_c2_cf_q_augmentation',
                'graph': case.graph_id,
                'space_group': case.requested_sg,
                'cf_sample_k': int(cf_k),
                'num_cf_q_samples': int(cf_stats['num_cf_q_samples']),
                'min_frac_rmse_before_qfit': float(topk_min([ev['frac_rmse'] for ev in raw_eval], 1)),
                'min_frac_rmse_after_qfit': float(topk_min([ev['frac_rmse'] for ev in fitted_eval], 1)),
                'best_after_qfit_rmse_to_GT': float(best_after['rmse']) if best_after is not None else float('nan'),
                'best_after_qfit_match': bool(best_after['match']) if best_after is not None else False,
                'best_after_qfit_valid': bool(best_after['valid']) if best_after is not None else False,
                'best_CF_NLL': float(cf_stats['cf_nll_min']),
                **cf_stats,
                'PASS': bool(len(candidates) > 0),
                'status': 'INFO',
            })
    else:
        rows_c1.append({
            'test': 'algorithm22_phase_c1_cf_score_only',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'num_q_per_template': 32,
            'ranking_rule': 'skipped',
            'top1_frac_rmse': float('nan'),
            'top1_rmse_to_GT': float('nan'),
            'top1_match': False,
            'top1_valid': False,
            'top3_min_frac_rmse': float('nan'),
            'rank_of_best_GT_candidate_by_CF': -1,
            'rank_of_best_GT_candidate_by_geometry': -1,
            'spearman_cf_nll_vs_GT_RMSE': float('nan'),
            'spearman_geometry_witness_vs_GT_RMSE': float('nan'),
            'cf_active': bool(CF_READY),
            'cf_mode': 'score_only',
            'cf_nll_finite': False,
            'num_cf_q_samples': 0,
            'num_cf_scored': 0,
            'cf_nll_min': float('nan'),
            'cf_nll_max': float('nan'),
            'cf_nll_has_nan': False,
            'PASS': False,
            'status': 'SKIPPED_CF_HEAVY_DISABLED',
        })
        rows_c2.append({
            'test': 'algorithm22_phase_c2_cf_q_augmentation',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'cf_sample_k': 8,
            'num_cf_q_samples': 0,
            'min_frac_rmse_before_qfit': float('nan'),
            'min_frac_rmse_after_qfit': float('nan'),
            'best_after_qfit_rmse_to_GT': float('nan'),
            'best_after_qfit_match': False,
            'best_after_qfit_valid': False,
            'best_CF_NLL': float('nan'),
            'cf_active': bool(CF_READY),
            'cf_mode': 'q_augmented',
            'cf_nll_finite': False,
            'num_cf_scored': 0,
            'cf_nll_min': float('nan'),
            'cf_nll_max': float('nan'),
            'cf_nll_has_nan': False,
            'PASS': False,
            'status': 'SKIPPED_CF_HEAVY_DISABLED',
        })
    init_rows = [('random_none', None)]
    if pyxtal_ref_ranked:
        init_rows.append(('pyxtal_top_geom', pyxtal_ref_ranked[0].candidate.q_init if hasattr(pyxtal_ref_ranked[0], 'candidate') else pyxtal_ref_ranked[0].q_star))
    matching_pyxtal = [c for c in pyxtal_ref_candidates if c.q_init.numel() == q_gt.numel()]
    if matching_pyxtal:
        init_rows.append(('pyxtal_best_raw_oracle_debug', min(matching_pyxtal, key=lambda c: float(torch.sqrt(cf_backend.torus_mse(c.q_init.reshape(-1), q_gt.reshape(-1)).clamp_min(0.0)).detach().item())).q_init))
    if cf_score_candidates:
        finite_cf = [c for c in cf_score_candidates if np.isfinite(float(c.cf_nll)) and c.q_init.numel() == q_gt.numel()]
        if finite_cf:
            init_rows.append(('cf_score_top_nll', min(finite_cf, key=lambda c: float(c.cf_nll)).q_init))
        if cf_score_ranked:
            cand_q = cf_score_ranked[0].candidate.q_init if hasattr(cf_score_ranked[0], 'candidate') else cf_score_ranked[0].q_star
            if cand_q.numel() == q_gt.numel():
                init_rows.append(('cf_score_top_geom', cand_q))
    if cf_aug_candidates:
        finite_aug = [c for c in cf_aug_candidates if np.isfinite(float(c.cf_nll)) and c.q_init.numel() == q_gt.numel()]
        if finite_aug:
            init_rows.append(('cf_q_aug_top_nll', min(finite_aug, key=lambda c: float(c.cf_nll)).q_init))
    seen = set()
    for label, q_init in init_rows:
        if label in seen:
            continue
        seen.add(label)
        out = fit_oracle_payload_from_q_init(case, state, f0_hat, payload, q_init=q_init, cfg=ALGO22_CFG, init_label=label)
        rows_c4.append({
            'test': 'algorithm22_phase_c4_q_init_stress',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            **out,
            'PASS': True,
            'status': 'INFO',
        })
    rows_c3.append({
        'test': 'algorithm22_phase_c3_cf_active_smoke',
        'graph': case.graph_id,
        'space_group': case.requested_sg,
        'cf_active': bool(CF_READY),
        'cf_mode': 'checkpoint_loaded' if CF_READY else 'none',
        'num_cf_scored': int(0),
        'num_cf_sampled': int(0),
        'cf_nll_min': float('nan'),
        'cf_nll_max': float('nan'),
        'cf_nll_has_nan': bool(False),
        'PASS': bool(CF_READY),
        'status': 'INFO' if CF_READY else 'NO_CF',
    })

safe_display_sorted(dataframe_result('algorithm22_phase_c1_cf_score_only', rows_c1), ['graph', 'num_q_per_template', 'ranking_rule'])
safe_display_sorted(dataframe_result('algorithm22_phase_c2_cf_q_augmentation', rows_c2), ['graph', 'cf_sample_k'])
safe_display_sorted(dataframe_result('algorithm22_phase_c3_cf_active_smoke', rows_c3), ['graph'])
safe_display_sorted(dataframe_result('algorithm22_phase_c4_q_init_stress', rows_c4), ['graph', 'init_label'])


[cf-eval-batch] subprocess start g2_cfscore_batch n=4


/workspace/.venv/lib/python3.11/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


,test,graph,space_group,num_q_per_template,ranking_rule,top1_frac_rmse_to_GT,top1_cart_rmse_to_GT,top1_match,top1_valid,top3_min_frac_rmse_to_GT,...,cf_active,cf_mode,cf_nll_finite,num_cf_q_samples,num_cf_scored,cf_nll_min,cf_nll_max,cf_nll_has_nan,PASS,status
0,algorithm22_phase_c1_cf_score_only,2,4,16,cf_only,0.013534,0.06104,True,True,0.013534,...,True,score_only,True,0,4,81.325653,81.325653,False,True,INFO
1,algorithm22_phase_c1_cf_score_only,2,4,16,geometry_first_cf_tiebreak,0.013534,0.06104,True,True,0.013534,...,True,score_only,True,0,4,81.325653,81.325653,False,True,INFO
2,algorithm22_phase_c1_cf_score_only,2,4,16,geometry_only,0.013534,0.06104,True,True,0.013534,...,True,score_only,True,0,4,81.325653,81.325653,False,True,INFO


,test,graph,space_group,cf_sample_k,num_cf_q_samples,min_frac_rmse_to_GT_before_qfit,min_frac_rmse_to_GT_after_qfit,best_after_qfit_cart_rmse_to_GT,best_after_qfit_match,best_after_qfit_valid,best_CF_NLL,cf_active,cf_mode,cf_nll_finite,num_cf_scored,cf_nll_min,cf_nll_max,cf_nll_has_nan,PASS,status
0,algorithm22_phase_c2_cf_q_augmentation,2,4,4,4,0.17826,0.009144,0.035466,True,True,NaN,True,q_augmented,False,0,NaN,NaN,True,True,INFO


,test,graph,space_group,cf_active,cf_mode,num_cf_scored,num_cf_sampled,cf_nll_min,cf_nll_max,cf_nll_has_nan,PASS,status
0,algorithm22_phase_c3_cf_active_smoke,2,4,True,checkpoint_loaded,0,0,NaN,NaN,False,True,INFO


,test,graph,space_group,init_label,q_init_distance_to_GT,q_fit_distance_to_GT,witness_after,frac_rmse,cart_rmse,match,valid,PASS,status
0,algorithm22_phase_c4_q_init_stress,2,4,cf_score_top_geom,0.303025,0.017748,0.002859,0.017743,0.071985,True,True,True,INFO
1,algorithm22_phase_c4_q_init_stress,2,4,cf_score_top_nll,0.303025,0.017748,0.002859,0.017743,0.071985,True,True,True,INFO
2,algorithm22_phase_c4_q_init_stress,2,4,pyxtal_best_raw_oracle_debug,0.313783,0.016070,0.002572,0.016072,0.053737,True,True,True,INFO
3,algorithm22_phase_c4_q_init_stress,2,4,pyxtal_top_geom,0.313783,0.016070,0.002572,0.016072,0.053737,True,True,True,INFO
4,algorithm22_phase_c4_q_init_stress,2,4,random_none,NaN,0.015593,0.002447,0.015593,0.069989,True,True,True,INFO


## Phase D
Local correction tests on fixed noisy states. Compare no correction, oracle-template, PyXtal-only, PyXtal+CF-score, and PyXtal+CF-q-augmented across weak alphas.


In [11]:
rows_d = []
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    for step in ALGO22_LOCAL_STEPS:
        state = sample_gt_noisy_state(case, t_value=algo22_step_to_t(step))
        f0_hat = clean_fractional_estimate_22(state=state, model=runner.model, config=ALGO22_CFG)
        before_eval = evaluate_arrays(case, f0_hat, case.gt_l_feature, case.atomic_numbers)
        witness_before = float(oracle_template_witness_22(state=state, model=runner.model, payload=payload, config=ALGO22_CFG))
        no_corr_update = kldm_cps_update_or_renoise_22(model=runner.model, state=state, f0_anchor=f0_hat, f0_projected=f0_hat, beta=1.0, mode=ALGO22_CFG.state_return_mode)
        no_corr_f0_after = clean_fractional_estimate_22(state=no_corr_update.state, model=runner.model, config=ALGO22_CFG)
        no_corr_after_eval = evaluate_arrays(case, no_corr_f0_after, case.gt_l_feature, case.atomic_numbers)
        no_corr_witness_after = float(oracle_template_witness_22(state=no_corr_update.state, model=runner.model, payload=payload, config=ALGO22_CFG))
        prepared = {}
        for source in [s for s in ALGO22_LOCAL_SOURCES if s not in {'no_correction', 'oracle_template'}]:
            candidates, cf_stats = generate_source_candidates(case, state, f0_hat, source=source, cfg=ALGO22_CFG)
            ranked = rank_candidates_against_clean_estimate_22(f0_hat=f0_hat, state=state, candidates=candidates, config=ALGO22_CFG)
            prepared[source] = (candidates, ranked, cf_stats)
        for alpha in [0.10, 0.25]:
            for source in ALGO22_LOCAL_SOURCES:
                try:
                    if source == 'no_correction':
                        clean_eval = before_eval
                        after_eval = no_corr_after_eval
                        witness_after = no_corr_witness_after
                        accepted = True
                        velocity_norm_before = float(no_corr_update.velocity_norm_before)
                        velocity_norm_after = float(no_corr_update.velocity_norm_after)
                        cf_stats = {'cf_active': False, 'cf_mode': 'none', 'cf_nll_finite': False, 'num_cf_q_samples': 0}
                    elif source == 'oracle_template':
                        branch = oracle_template_projection_22(state=state, model=runner.model, payload=payload, alpha=float(alpha), beta=1.0, config=ALGO22_CFG)
                        clean_eval = evaluate_arrays(case, branch.f0_projected, case.gt_l_feature, case.atomic_numbers)
                        after_eval = evaluate_arrays(case, branch.f0_hat_after, case.gt_l_feature, case.atomic_numbers)
                        witness_after = float(branch.witness_after)
                        accepted = bool(branch.accepted)
                        velocity_norm_before = float(torch.linalg.norm(state.v).detach().item())
                        velocity_norm_after = float(torch.linalg.norm(branch.state_candidate.v).detach().item())
                        cf_stats = {'cf_active': False, 'cf_mode': 'none', 'cf_nll_finite': False, 'num_cf_q_samples': 0}
                    else:
                        candidates, ranked, cf_stats = prepared[source]
                        if not ranked:
                            continue
                        branch = algo22_backend.branch_survival_step(state=state, model=runner.model, projected=ranked[0], alpha=float(alpha), beta=1.0, candidate_pool=tuple(candidates), config=ALGO22_CFG)
                        clean_eval = evaluate_arrays(case, branch.f0_projected, case.gt_l_feature, case.atomic_numbers)
                        after_eval = evaluate_arrays(case, branch.f0_hat_after, case.gt_l_feature, case.atomic_numbers)
                        witness_after = float(branch.witness_after)
                        accepted = bool(branch.accepted)
                        velocity_norm_before = float(torch.linalg.norm(state.v).detach().item())
                        velocity_norm_after = float(torch.linalg.norm(branch.state_candidate.v).detach().item())
                    rows_d.append({
                        'test': 'algorithm22_phase_d_local_correction',
                        'graph': case.graph_id,
                        'space_group': case.requested_sg,
                        'step': int(step),
                        'source': source,
                        'alpha': float(alpha),
                        'clean_frac_rmse_before': float(before_eval['frac_rmse']),
                        'clean_frac_rmse_after_projection': safe_metric_float(clean_eval.get('frac_rmse')),
                        'clean_rmse_to_GT_after_projection': safe_metric_float(clean_eval.get('rmse')),
                        'clean_match_after_projection': safe_metric_bool(clean_eval.get('match')),
                        'clean_valid_after_projection': safe_metric_bool(clean_eval.get('valid')),
                        'post_frac_rmse_after_return': safe_metric_float(after_eval.get('frac_rmse')),
                        'post_rmse_to_GT_after_return': safe_metric_float(after_eval.get('rmse')),
                        'post_match_after_return': safe_metric_bool(after_eval.get('match')),
                        'post_valid_after_return': safe_metric_bool(after_eval.get('valid')),
                        'witness_before': float(witness_before),
                        'witness_after': float(witness_after),
                        'accepted': bool(accepted),
                        'velocity_norm_before': float(velocity_norm_before),
                        'velocity_norm_after': float(velocity_norm_after),
                        'improves_witness': bool(float(witness_after) < float(witness_before)),
                        'improves_frac_rmse': bool(safe_metric_float(after_eval.get('frac_rmse')) < float(before_eval['frac_rmse'])),
                        'improves_rmse': bool(safe_metric_float(after_eval.get('rmse')) < float(before_eval['rmse'])),
                        **cf_stats,
                        'PASS': True,
                        'status': 'INFO',
                    })
                except Exception as exc:
                    rows_d.append(error_row('algorithm22_phase_d_local_correction', case.graph_id, exc, 'local_correction', space_group=case.requested_sg, step=int(step), source=source, alpha=float(alpha)))
                finally:
                    notebook_cleanup()

df_d = dataframe_result('algorithm22_phase_d_local_correction', rows_d)
safe_display_sorted(df_d, ['graph', 'step', 'source', 'alpha'])


[cf-eval-batch] subprocess start g2_cfscore_batch n=1
[cf-eval-batch] subprocess start g2_cfscore_batch n=1


,test,graph,space_group,step,source,alpha,clean_frac_rmse_to_GT_before,clean_frac_rmse_to_GT_after_projection,clean_cart_rmse_to_GT_after_projection,clean_match_after_projection,...,cf_active,cf_mode,cf_nll_finite,num_cf_q_samples,PASS,status,num_cf_scored,cf_nll_min,cf_nll_max,cf_nll_has_nan
0,algorithm22_phase_d_local_correction,2,4,600,no_correction,0.10,0.004279,0.004279,0.018199,True,...,False,none,False,0,True,INFO,NaN,NaN,NaN,NaN
1,algorithm22_phase_d_local_correction,2,4,600,no_correction,0.25,0.004279,0.004279,0.018199,True,...,False,none,False,0,True,INFO,NaN,NaN,NaN,NaN
2,algorithm22_phase_d_local_correction,2,4,600,oracle_template,0.10,0.004279,0.004547,0.019145,True,...,False,none,False,0,True,INFO,NaN,NaN,NaN,NaN
3,algorithm22_phase_d_local_correction,2,4,600,oracle_template,0.25,0.004279,0.005075,0.021894,True,...,False,none,False,0,True,INFO,NaN,NaN,NaN,NaN
4,algorithm22_phase_d_local_correction,2,4,600,pyxtal_cf_q_augmented,0.10,0.004279,0.004147,0.017621,True,...,True,q_augmented,False,8,True,INFO,0.0,NaN,NaN,True
5,algorithm22_phase_d_local_correction,2,4,600,pyxtal_cf_q_augmented,0.25,0.004279,0.004383,0.018902,True,...,True,q_augmented,False,8,True,INFO,0.0,NaN,NaN,True
6,algorithm22_phase_d_local_correction,2,4,600,pyxtal_cf_score,0.10,0.004279,0.004351,0.018425,True,...,True,score_only,True,0,True,INFO,1.0,117.032242,117.032242,False
7,algorithm22_phase_d_local_correction,2,4,600,pyxtal_cf_score,0.25,0.004279,0.004893,0.021334,True,...,True,score_only,True,0,True,INFO,1.0,117.032242,117.032242,False
8,algorithm22_phase_d_local_correction,2,4,600,pyxtal_only,0.10,0.004279,0.008887,0.044577,True,...,False,none,False,0,True,INFO,0.0,NaN,NaN,True
9,algorithm22_phase_d_local_correction,2,4,600,pyxtal_only,0.25,0.004279,0.020367,0.103783,True,...,False,none,False,0,True,INFO,0.0,NaN,NaN,True


## Phase E
Lightweight actual-sampling comparison against the KLDM-PC baseline. Use the same late weak schedule and compare the methods that matter.


Laptop mode: this phase uses cached replay results and the minimal default source set needed for a scientific comparison.


In [12]:
rows_e = []
rows_e_top = []
for case in GRAPH_CASES:
    baseline = replay_source(case, source='baseline', cfg=ALGO22_CFG)
    rows_e_top.append({
        'test': 'algorithm22_phase_e_best_iteration_compare',
        'graph': case.graph_id,
        'space_group': case.requested_sg,
        'source': 'ground_truth_reference',
        'step': 0,
        'best_step_frac_rmse': 0.0,
        'best_step_rmse_to_GT': 0.0,
        'match': True,
        'valid': True,
        'beats_baseline_best_step': True,
        'status': 'INFO',
    })
    for source in ALGO22_PHASEE_SOURCES:
        replay = baseline if source == 'baseline' else replay_source(case, source=source, cfg=ALGO22_CFG)
        rows_e.append({
            'test': 'algorithm22_phase_e_sampling_vs_baseline',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'source': source,
            'mean_frac_rmse': float(replay['frac_rmse']),
            'mean_rmse_to_GT': float(replay['rmse']),
            'match_rate': float(replay['match_rate']),
            'valid_rate': float(replay['valid_rate']),
            'group_witness': float(replay['group_witness']),
            'accepted_fraction': float(replay['accepted_fraction']),
            'projection_count': int(replay['projection_count']),
            'best_step': int(replay['best_step']),
            'best_step_frac_rmse': float(replay['best_step_frac_rmse']),
            'best_step_rmse_to_GT': float(replay['best_step_rmse_to_GT']),
            'best_step_match': bool(replay['best_step_match']),
            'best_step_valid': bool(replay['best_step_valid']),
            'improves_witness': bool(replay['improves_witness']),
            'improves_frac_rmse': bool(replay['improves_frac_rmse']),
            'improves_rmse': bool(replay['improves_rmse']),
            'beats_baseline': bool(replay['beats_baseline']) if source != 'baseline' else False,
            'PASS': True,
            'status': 'INFO',
        })
        rows_e_top.append({
            'test': 'algorithm22_phase_e_best_iteration_compare',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'source': source,
            'step': int(replay['best_step']),
            'frac_rmse': float(replay['best_step_frac_rmse']),
            'rmse': float(replay['best_step_rmse_to_GT']),
            'match': bool(replay['best_step_match']),
            'valid': bool(replay['best_step_valid']),
            'beats_baseline_best_step': bool(float(replay['best_step_frac_rmse']) < float(baseline['best_step_frac_rmse'])) if source != 'baseline' and np.isfinite(float(baseline['best_step_frac_rmse'])) else (False if source == 'baseline' else False),
            'status': 'INFO',
        })

df_e = dataframe_result('algorithm22_phase_e_sampling_vs_baseline', rows_e)
safe_display_sorted(df_e, ['graph', 'source'])
safe_display_sorted(dataframe_result('algorithm22_phase_e_best_iteration_compare', rows_e_top), ['graph', 'source'])


,test,graph,space_group,source,mean_frac_rmse_to_GT,mean_cart_rmse_to_GT,match_rate,valid_rate,group_witness,accepted_fraction,...,best_step_frac_rmse_to_GT,best_step_cart_rmse_to_GT,best_step_match,best_step_valid,improves_witness,improves_frac_rmse,improves_cart_rmse,beats_baseline,PASS,status
0,algorithm22_phase_e_sampling_vs_baseline,2,4,baseline,0.004296,0.018215,1.0,1.0,0.003006,1.000000,...,0.002140,0.009221,True,True,False,False,False,False,True,INFO
1,algorithm22_phase_e_sampling_vs_baseline,2,4,oracle_template,0.004437,0.018731,1.0,1.0,0.000036,1.000000,...,0.002662,0.011007,True,True,True,False,False,False,True,INFO
2,algorithm22_phase_e_sampling_vs_baseline,2,4,pyxtal_cf_q_augmented,0.004380,0.018380,1.0,1.0,0.000736,0.166667,...,0.002035,0.008142,True,True,True,False,False,False,True,INFO
3,algorithm22_phase_e_sampling_vs_baseline,2,4,pyxtal_cf_score,0.004291,0.018072,1.0,1.0,0.001221,0.666667,...,0.002500,0.010243,True,True,True,True,True,True,True,INFO
4,algorithm22_phase_e_sampling_vs_baseline,2,4,pyxtal_only,0.004291,0.018072,1.0,1.0,0.001221,0.666667,...,0.002500,0.010243,True,True,True,True,True,True,True,INFO


,test,graph,space_group,source,step,best_step_frac_rmse_to_GT,best_step_cart_rmse_to_GT,match,valid,beats_baseline_best_step,status,frac_rmse,cart_rmse,PASS
0,algorithm22_phase_e_best_iteration_compare,2,4,baseline,750,NaN,NaN,True,True,False,INFO,0.002140,0.009221,False
1,algorithm22_phase_e_best_iteration_compare,2,4,ground_truth_reference,0,0.0,0.0,True,True,True,INFO,NaN,NaN,False
2,algorithm22_phase_e_best_iteration_compare,2,4,oracle_template,750,NaN,NaN,True,True,False,INFO,0.002662,0.011007,False
3,algorithm22_phase_e_best_iteration_compare,2,4,pyxtal_cf_q_augmented,750,NaN,NaN,True,True,True,INFO,0.002035,0.008142,False
4,algorithm22_phase_e_best_iteration_compare,2,4,pyxtal_cf_score,750,NaN,NaN,True,True,False,INFO,0.002500,0.010243,False
5,algorithm22_phase_e_best_iteration_compare,2,4,pyxtal_only,750,NaN,NaN,True,True,False,INFO,0.002500,0.010243,False


## Summary
Compact pass/coverage summary for the current Phase A–E pipeline.


In [13]:
focused_keys = [
    'algorithm22_cf_checkpoint_load',
    'algorithm22_phase_a_default_replay',
    'algorithm22_phase_b_pyxtal_feasibility',
    'algorithm22_phase_c1_cf_score_only',
    'algorithm22_phase_c2_cf_q_augmentation',
    'algorithm22_phase_c3_cf_active_smoke',
    'algorithm22_phase_c4_q_init_stress',
    'algorithm22_phase_d_local_correction',
    'algorithm22_phase_e_sampling_vs_baseline',
    'algorithm22_phase_e_best_iteration_compare',
]
summary_rows = []
for key in focused_keys:
    df = result_tables.get(key)
    if df is None:
        summary_rows.append({'table': key, 'rows': 0, 'pass_rate': np.nan})
        continue
    pass_rate = float(df['PASS'].mean()) if 'PASS' in df.columns and len(df) else np.nan
    summary_rows.append({'table': key, 'rows': int(len(df)), 'pass_rate': pass_rate})
display(pd.DataFrame(summary_rows))


,table,rows,pass_rate
0,algorithm22_cf_checkpoint_load,1,1.0
1,algorithm22_phase_a_default_replay,4,1.0
2,algorithm22_phase_b_pyxtal_feasibility,9,1.0
3,algorithm22_phase_c1_cf_score_only,3,1.0
4,algorithm22_phase_c2_cf_q_augmentation,1,1.0
5,algorithm22_phase_c3_cf_active_smoke,1,1.0
6,algorithm22_phase_c4_q_init_stress,5,1.0
7,algorithm22_phase_d_local_correction,20,1.0
8,algorithm22_phase_e_sampling_vs_baseline,5,1.0
9,algorithm22_phase_e_best_iteration_compare,6,0.0
